# H2O feature tables (STO-3G): cluster run

Run the cells from top to bottom. The notebook needs nothing but Python 3.11+ and `git`:

1. it downloads the repository (`Zephrous5747/ADAPT_Sampling`, branch `part2`) and installs the packages,
2. it writes the six feature-study scripts into the repository (they are embedded in this notebook),
3. it runs the H2O calculations (equilibrium 0.9572 Å and stretched 1.75 Å, STO-3G, UCCSD pool),
4. it makes the three tables (LaTeX + CSV) and their figures.

| Table | What is computed for H2O |
|---|---|
| 1 | correlation of the gradient estimates from the shared measurements vs the energy lowering of each generator, on the HF, CISD and every exact ADAPT state |
| 2 | complete measured ADAPT runs (II-A) at 99, 90 and 68 % confidence, 2 runs per level and geometry (12 runs); 95 %: the 36 recorded runs of the repository, with their optimization recomputed exactly |
| 3 | optimization cost of runs with selection errors: the certified II-A runs and uncertified fixed budgets (recorded runs, optimization recomputed) |

**Time and memory** (one job = one core):
- Table 3 and the 95 % row of Table 2: minutes to about an hour per job, < 1 GB each.
- Table 1: about 35 jobs (one per state); with the II-A design up to about an hour each, 1-3 GB.
- Table 2, new levels: **one measured H2O run takes 8-16 h and about 5 GB**; 12 runs in total.
  - `MODE = "local"`: all runs in parallel on the node that runs this notebook
    (12 cores and 64 GB run everything at once, about 16 h).
  - `MODE = "slurm"`: the notebook writes a SLURM job array (one task per run) and submits it;
    run the status cell later, and the table cell when all tasks are done.

**Resume.** Every step skips work already on disk, and measured runs checkpoint every selection.
If you stop the notebook (or the computer sleeps), run the same cells again: the work continues.
Set `DRY_RUN = True` to only print the commands.

In [ ]:
# ---- settings (edit) -------------------------------------------------------------------
from pathlib import Path
import os

WORKDIR = Path("~/adapt_h2o").expanduser()       # where the repository is downloaded
REPO_URL = "https://github.com/Zephrous5747/ADAPT_Sampling.git"
BRANCH = "part2"
CASES = ["H2O_eq_HF", "H2O_stretch_HF"]
DRY_RUN = False                                   # True: print the commands only

def _ram_gb():
    try:
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
    except (ValueError, OSError, AttributeError):
        return 8.0
CPUS = os.cpu_count() or 1
WORKERS = max(1, min(CPUS, int(_ram_gb() // 3)))       # Tables 1 and 3
WORKERS_TABLE2 = max(1, min(CPUS, int(_ram_gb() // 6)))  # measured runs need about 5 GB each

# Table 1
TABLE1_LEVELS = ["II-0", "II-A"]       # II-0: fixed fragments (fast); II-A: learned splitting design (slow)

# Table 2
DELTAS = [0.01, 0.1, 0.32]             # 99, 90, 68 %  (95 % comes from the recorded runs)
RUNS_PER_LEVEL = 2                     # measured runs per level and geometry (each 8-12 h)
MAX_HOURS = 22                         # a run starts no selection after this; run it again to continue

# how the Table 2 runs are executed
MODE = "local"                         # "local": this node; "slurm": one array task per run
SLURM_TIME = "24:00:00"                # wall time per task (keep MAX_HOURS about 2 h below it)
SLURM_MEM = "8G"                       # memory per task
SLURM_EXTRA = []                       # e.g. ["--account=xxx", "--partition=yyy"]

# Table 3: uncertified fixed budgets (shots per selection, pilot allocation) to recompute
FIXED_BUDGETS = {
    "H2O_eq_HF":      [10000000, 23713737, 42169650, 56234133],
    "H2O_stretch_HF": [31622777, 100000000, 177827941, 316227766],
}
FIXED_MAX_RUNS = 12                    # recorded runs per budget (36 = all; more runs, more time)
print(f"{CPUS} cores, {_ram_gb():.0f} GB RAM -> WORKERS={WORKERS}, WORKERS_TABLE2={WORKERS_TABLE2}")

## 0. Download the repository, install the packages, write the feature scripts

In [ ]:
import subprocess, sys, importlib.util

WORKDIR.mkdir(parents=True, exist_ok=True)
ROOT = WORKDIR / "ADAPT_Sampling"
if not (ROOT / ".git").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(ROOT)])
REPO = ROOT / "part2_estimator_design"
assert (REPO / "src" / "learning.py").exists(), "unexpected repository layout"

needed = {"numpy": "numpy>=2", "scipy": "scipy", "pyscf": "pyscf", "openfermion": "openfermion",
          "pandas": "pandas", "matplotlib": "matplotlib"}
missing = [spec for mod, spec in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
assert int(np.__version__.split(".")[0]) >= 2, "NumPy 2 or newer is required (numpy.bitwise_count)"
print("packages ok; repository:", REPO)

In [ ]:
# ---- the feature-study scripts (embedded; written into the repository) ----------------
import base64, zlib
FILES = {
    'scripts/features_energy_lowering.py': 'eNqlWm1v2zgS/u5fwdN+kbq2YqfF4c67Llp00xdgey12uz0cDK9CS7TNRpZUUU6apvnv98yQkmg7Se+lwG4kcjgznPcZ+Ye/nOxMfbLUxYkqLkV13WzK4vEgCIKXSja7WgnT7LLrqdiUV+JK5bloNkqsa5lpVTTi2zrR30RVq0ynjeE9Vah6fS2kWNOTbMpayLTZyTy/FmupCxMPBi+xqC5VfQ3sslGiXPHRpSrSzVbWFyKk19cvhSwy8eLN779YOEOASqYb4C63qsF5AmBMgz1M6gtoiue/PH//QTS1/KRS8HEd9eAed+7IHy9egE5VljnuKdJyW+0aNRzowiGTtW42IKrT4WDwSJyf4+piJn6ujP42fz0UrxK9+IaXp+fnQ97PziaJPj+f+lLJyytV62JNRKUoCzWqZC2BVdViW14qHNvqIsGJRoqzUP15Yx+B/FYAeSRGWKYHoiIca0uVyp1RwLhS9VaXhU6xk2pIBC/eVQ0WzEpDkOfnr/58DPZHr4BImBK4zs97crfYmogfhdFFyEuReIXXcAL6aWncGhb/PD0//8ldd7XLc7oxtNeJ3CluKMqKeJF5tCcPuaKby6pSRUZiAVuMoSkJCkylu7omQ5MFeP/K+qvViJBttaETVpudFIcC8ttZS2zPWlowu/egZc0EGq5VVdYNA9ayuMC1AJ1biUE739bfxBU0zgCt2obiChffAA2vynqtTDMgUG2EzE3pr3uniG3aMRtZe9YOmKo0Ri/zngS2JHM1cGhGhD7dlDpVHZ6rssbZelOO1mWZtbuh9canM6snbEdiK7/gfCTKZQPfgwze7RoY9hSSrneFOVlZLzcnVkhJy0bSGY2JU3NJSoW1ihpRoOqkSOx0cNGAXr+H1prDvSgjcCjwz8YhYdJaV02PLjlAF1fXYjQiuxvVCkYhXj9JzOcdhJyoz4nRmZpU4wRx5Ff9OvntMT9TbBus6nIrkmS1I7RJIvSWjUEWRWmdxgwG7Vq9hnUZ1b6D9fbRXJv2EfaoLNZKNptcL1uU7/Ha4Sp2W3AsjSgqC2xSXV3HdHXTHjCVkvVWFvVgAPwxoYsRNVXdhGN4alOHhDIE8zoH61EMuZT5pQojwJK9m/lkIU5EYOo0iKKONjabSbKsdbZWQvwgivKznIqzJ+PT4csn40nLvAfkDq5Vk6TSHByy8J6jO+gXr8/evnnx/Nfk+YsXf/z2/MW/huJ5Jqvm92vTqO1QwDwSSQsH6Aavzt69Pfvw25uz3xF7btgKgnvUGUxFiD0xicfB8B4oyhpBNDzEA/mpJt0w2OkestNDZIegexg9g2IEeBePGcM/YNQd3dN3xFNL5PSdUJ+ZiF3f5xFrLUnvAJbaE+1ue+x2MBhkaiVyvd40iWEBh0toitU1paPIF099+U8trSA4P/dW4+VO5xkckgJeubNB8b3c5brP9FVdIkxtRQgPEYVSmcoEAqH6ifJBg9y+Qn4Bl1FM/kVU2EDSjUKgbnr7YFLJ591SN8lGbnXeIF3JYiguZa4zhAB/lWzkPUxSvOkxcoZ2yHZparKEVvp9Vym05KTRJtFFpr7AedhUk2VephfdG/YQPI2NO3C+FObXmnwvzYi3wRp277lCSIct3F134e0hobAwji7wFVUsjaxreR3u8xQCOC4sJcNH8SbzaiPblyWl4aHImutKzYBHF81fn1j8fEtg9y/NCMvKBuwWR4veEaWAIvO4KVNThxYVi3zmSftOxlQOSjVCpz3UJxAcne9xsY6t8P4rTtjA1thjdhZWHysrvq+qRlHijsVGf231tZpj1yCgphsDc1BZCzT0LSNEAbBW4eE19tmKogUVRfHYmlpKxd8qL2UTgkSuUdqsY6XXUL3ZhHxPcG61GkXz8cJyZL0URz33663sUBAHHHWSGQpnw72QbRS58998HedyqfIDCQL9akgXGYqboN6sXHJF5CGy/cKtz3mcWB/A/3kVEWlXF27TxaNUw0wuWeGh3Zju5wAG2I9RkGGRsbSm97kyuozKxmWqPfrIwPAJboIyAJztgYUe14iiNnJaF2VHB1mysZDOzh2sk7FTmLsggZ+IXtNFWW/DS/J3e2cE/aQrQrv65O7ro3yfehfm+ze7KlfzfnHoASy6qH3G5f5+j+DKc+oLxugKuCiD8VDdAjDW+WHHE/rVfx+w1ZjChRWCVT13GrzHVFywgghMJVMVjvBSaWaV/jwZjyfOVqDjiQXuOgjyJ3gPGMVq10O4YOGXyuu8jYrWrXNVtGrszT2Kht+FsIaE+4I3NnuF+ot21R3Q085/LonztXhGeuoXT93i5aRb06sjg5hE4mcxUaPJk+meP6Zl0aAvUd0i8iJHZ0Z5itbqQbwMHYmnjHq8j7qWGrXZR5nv1Fldw+VWQa/pG7SNWQl5Uta23d+1aHUPo4Ve0bmwqzi/oH8fneoaiWA9hxiQnSf47xQxQ37RZjbpYd8C9mP8Addohern72fiYw+almq12kONJziOSXJ9AZ1ApWw4i4i8+7H4IoruLNujbg1DwQJ32zBA3SCXw2UxelqgSGL8Q/HWPfWESYb/yUl7j49H5z3i3eOJxdrBfMImsi8lA+t5YQsa9Yh+QDhZaTQ9si53rpdbo9implVvd9sO0GsObK/7tSvHGRTviUklWsRBd4bC0CpsogPLc2HOZqI5PHDYueWRR0aLaN+2bPCzaQ6YnkHdz4AywvW9xdAJLrKbPUt5iSRDidL6+hzdaPgJ9GBM4wjG1C6DmU/wASyTK7vAQHDRYtDzQvI/uH0IrS1JlGYWWmK4E2LwpsxmAW+oLGiHD2Z2E3yBW+RIceRIp7f9bbkZZ/QhCMWrHSrSVoHzT55Y2kg11wsHz0dHiJ64VDzuIW0ks3CE8ws5tkMufp756IXK4cNOGp8WfuI5jIwu3/CshXve7ySbsjJTdAimmcM6IXJOUsbPP7joeGrVeX8RkSKtcElrjlLXYeqmBoJNjni8Jzbb9CYLL0ZTgPaodJwkw3ZY1CUnN/5RId0rxA0jGA/dL9RkVWTtPFEK7WWtXnrFgMFeew73gf6c8AHp5I2gsZU106RSiguXobhArdA+Tsn9xTfu/5Bv9iWMnvRwgaSzt3Yg+3pTtlpxAkgUxXe3yKKnqauVlFyatXP0pQnXrgSHv2GR9mJyPXe38oo67MAKE77AN0J1ZNPAlG+F1ws8BgEZ7QWNt+he1k5RdwY+R8n2tWSXeiwe7bG6d6UAHCRgJVkDlji77XRfoGwikZAJhGEwqQKWGKwkDEhMQSuvyAtuNktKmBj1PQbGFGYqOoh+hP1CXTPioJ2sUD9NTpusE3QA/JSd0SKP5+zCun/nYVsCdSQ0bOsOZ2d87/tdJijuPKS95QNunXbmq+AGTN8mNySY24BM1ZVo99YTW2kunKptmfC3bouG7pTvMy6aq9LoRl/2A8e9EJg4YOudZDO00LtOU1aJs7NVLpuiLKgCCy3ZmTW4ka1SOIn741CaY4YN5U5dpPkOkbnH20rkftT9SPORT+bUw8GSa5XsC8+mqm6wxjjnJDEEC7qffY54DoeIopHfYF20GFOxQIXXqTV9Tw0sLnXMMEsQDPfiZE7/fsjokQX6HC+pybb9vKqpd55kIct+6OhG3OWCsfGRADwjPhbCvmLnjHNB6bxjl+/eMw8KR1f3Ce17x70EEWmZYAv4P9L0PM8nZUNPh+DgUHEne5Qu2nVvZHBMS999/i4VtXD3acmlFeB3aWULbkMO5RRebTTgaXNNzuwmz/Hzeo3GpWje8w7CnJ2Lo6SZJUlWpkkSeSdjmWWJdEfCYDSiTtsgBBVYNLPgRxtO5S5vZpxA+8lr9CCefs7eI3vkIZsv7g+GG5VXs8B9tqMYcLUpqeQ5/FBkv6KslZ1Bckvff++xH2fawBU8yCxUAdZ4LuaSaMvmOJ48eBJJvz1Jk/b+IM/dg73vG4GTGAmDZg8WI/8hnCbst2PgjbcXma5DN6effah3yHsKHVWTlBf82s3Okopps1T42RoEo6Hp/sPfa3CBh4D7rzBBRw9GaVp69ExTO8TH+aJL0/wRAO7JmNmo+vTlaohusoPTvVnNaaWv5FM3maRvJnGl6lWSolKHF4W9C3Zzsr3hdj+JZRhrMeAzDF6/dKP3YVsk0nBrvvCmBCgBF3710POqbX/cu+A+hbaaDO38qKVzPOby7h+1UPw5wGsleYIz6z+DdEfRl0CBRapmR59Q+uOkBoBXov0sPZ9MR5MFXYcKbV7y0tV8sX8ftPJ9Fe0GZMCGgppVjydbMUcPSoE/qwfugG7I7OCXPNdyi0f49m9gC8wLd4KBLRxdy1Lb53vdc91+kDD9bKqT7QPzq75Jnri2E7APDu2Yu8PDbsbomhs7t2CUfjvj25jvMn3PxkM63Pt+y3PnSanc38xmYjy9M7p2TIEPdPHVdRgdwZExfOf0/Q3lnoaG3CcdlDxd1+FKzyg6KsK5vbNjdrrTOrqj9G2jUGtp/2eL0kVEAOkHxuNdvc5EAOxMiF+pfweutm/ha2IJ7h3Q7zv8zgfyJ+D/gBDb3ySYWkO0FNzPJyxCeyevr+H5NMja27X03OrtHe66J8iD5nXYO+B6aD3C4nJJA6mT1Iy61UlilWr3LcDTbFVTBYVWhXHeTsUNt/rsvdFt97uPm7sHALf+96HgQGarILy5IznQb08odUzj8erWRIhAq3xnNl7e5HnBVY2YFPYptDWrI5A+t/p5r53oOyDangpbCNC2G6ZQ5704KNzQsxnOWKipaD+mKR4thhfsAzX7AKVWDoL85gTKvzThT/0lNBYGV1RhqatcF2oWBBH9ZGCDmJF7XszsUZmILB7/AnL/5IXQwkE2WuUZlatmRjxEBwdj/rNRMtvLu/4msRq2IoE9JgmhSxIKRkGSUO2aJIHlyBayg38DIpYQQQ==',
    'scripts/features_gradient_correlation.py': 'eNqtWW1v20YS/q5fsWWBKxnTtJwUuDulKs5wnDpA84Ik7eFOEDYrcSVRokiGS8lWXfe33zOzyxfZktsPJ8ASuTs7MzvvM/72m7ONKc8mSXams60odtUiz170PM97rVW1KbUw1SbeDcQ0L0udqirJM5HPRLXQYl6qONFZJbSpkrWqtBG+WahSx2KtlcHhNXZNINRcJZkBXKbL+U6k+Y0uk2we9XoXadqgMQJHG1yxmJX5mukYtcaXRTzbpOkOzKzXmwoo8JRV+rYyoTA5ASdlr+WG8NV86zgS4nVeCr3V5Q63wpLwr1+H4vLNp1dCZfHejruivlXTSly8uvjwuVeVaqmnVV7uAnFDiNfFptJhy6cFzjN9ahZ5BYCtKhOVTcGJQ8e4w17vGb+1AAL8lsntE4KdgXPGYMUQa5PMs54Q/ps3p/2BmCW3LDE1Z5G/FFi+GIi8VNOUCOnZLJkySlOkSUWiC0Jxk1QLhzWvmEkwD6QqTfOpUzVkQu8iy7PT33SZQ6hr6Nny35pEnIPJDLeOdaEhSywRSJVXKg1e4sL1BVKtYqD0U1XOcT3x+/z3oCP96QLEdDYHyHIgHlJx8qlucnD5TE1MnkIFzw4ILBSlGAojU/wtxWW+9ecyDcVcLgNxBjONaSfGW0jEgW26SHDgV1X6dAjA4pSP1ieww6snwj5hdSjOAfQclBp0DrKGA9TA4f4BwPCKzOypUVUKul/pjmhOOyIgG4OJGFx9utCq0CWp3GqLBKqc4uEFsyTWZElmqqBx1izIsuhb23zggCRPxQYLKnCyCiTX0HhrGE5bpB9eW7K4aAuM8CVqnVjI70yLvFqoSiS0ADUnM7Gkl+kiNzoTFA5wABHg/aaCGw3Ely/lJjNnMxt1zFmtUtnRvyxUUppoarZfvggffIsyvxGQinNaYq2VXgDsAp/3bz/Id7+8lZ+vP15dvPo0PHdRDqIqk6JqacpDNKNiJ05Pp8rAvq+/l+brBpeW+qs0EPh50ZfXryla9jgKSDnbECopRbIu8rICS3AKRmR6vXqtnEPeRtfvuE/9aHamfoQha4u1UNUiTSY1yg94bXBlmzUYVPC9ote7vvp4BaskAB+sJCkYCSLcLE+32g8iUMXteiASEc4IWtBl5fcRPKvSp9MOBGbsmXLqBcFTwNit2cCx6lxOyiSeayG+RST4qgbi6vv+8/D19/1ze49OWKklYVceHLHQdWSvYSebJI2lW5WQR6kQLw4ctLGxPvaK3z7pKnQY6iApizKfpBrB7AAOpJwihV05JO85jr7NObgegm9zQ33k8vrq7ZvLi5/lxeXlLx8vLv8TiotYFdWnHSx/HQpHXSowBsuXijYfoLa4G/O03isbB3OUfrp6//bq88c3V59CMU0QhbbMSUiOLRvHbo89oPHu/bv/Xn18T9FMn/6j1+vFekZBXk6gm5Xxi0Cc/ijiZFqNkgzMVhtIZpQVURarslS7ULTP4/GAXQ4OcWkVRWf9D2qTJiKJEZU7qQiFgc04RA2Bw+otImciHPmmAk939/xCwW6F2JMWC4XggUC2WesSluMX0bS6lcAdWNL0SfNQcDi3m0VVjlbjsPuG8Hw+buBXWheAxj2QUoDxdpTmg0UyDsSPot9AIYQRYKSynd8h5ngl4fjMXzAGLuIrz8sYCsDdHb4RnWdGbvdWAkZWaqg5I1xOCW19IG194FsRDbpGTel4INLEMAdAvrZWGtqkPhBdTVGQHkB+Feu03Wm09mu3YrEZGQRsYH1czrjswJmhBfZz7KHOysAyYFM9qwRVDkGjWmtZkNKoY2dO/bVXjkj8pPaE9E23tPrKcAzx3acVKzcwZpVHRIzvZ7DIwG5xviA6voKkApczga9UyBB+BlWdYJPugfBg9yfdfVL6RHwz5KuOG1MENgZj/IOuhXwuN3rfNrgEAg/9qL+3znhqeza68q0cRgr3/lt3YTJ+YG2OFOt3xCjG4gcQeAxlxZOh3NvoR5sFbnGrwFlD2CF7DDkB5KSFnByFhB9aXYAqBfaMXHRElIqJM/Pu5xLAzl6j1r6sG4WE7PERK86ToZiluap8XOBf4nI0wG0ik/yGctw9DMbYuJ1Q7dYVVG9fNNsRqZIclp5xTUXPTKPrk9h0Pkkal6g6DArYiUZFuUoy2M6qCeihi2PWMzvuSD0QeeBWp06fAAMxZ/f4WivTMWtb5Q2bHOkDIhTnUd9uf9smyBuqqdriXywUSjifI2SCajBFv8KVe9NtzHUl/uhbCi9JkLY2X9seSlWOQP+sjwJrk8Wn+QzRiEsZFPrrJOPeibst5qJMJhub1rm6zcQfSCXnf4fo8fvPJlrYin//djCVtbpN1pu1LWtDe+SZBaA93/kyycrJOGpaxlEbGBC6520Qn9tD5LZYpNhMG+WcEBJk0I0dxwLtIx06Fgmru4alSMA1STIOCjlttFi2EaWJXMFe1FiKoQ0xg96TrruFwW2Xzlpt1OLAFPICivNla+C2EYB/4cZWyvQTJ2ruAzgIWNIv+v2gywh061siAYUUUmOflEG4BmQTKmt6LBv39+0sa61sgL4I9gDz0bMk0/Hezag5sx0XJSGVNZs6NQ/iJ/dxfNkxcTJa0k97f7yesQq+ojjdptgD7/tSZDq0d0LCo47t2dMYueFz8C2uOfWPJHzyZDY8i2McdlaW41YDZAmRKqgf9u88w4Wfh1xtA4fHPQvebQTxVnj0PK4ySG7vqDAiaVBs8aw90mGKH+HBQC88m48B1TKHs21HVO8sx8cwtLASBkssOdcxcFD2sB/Z0UhmJ2wez4OAno4hBLBE29wwRmdrxuzePnsW9zFsc1VINGYA9FtM0Kg7RnrrUDh2RzR3EmDAUrJ4EjzhG89bat1rXskPeGWPw+3yOHNJ5k7L9bUC7Ll+Acsiu7GbHenuARxF2VT8xtT3ntNgYs4zCTzCWvCNKtWainWmY9g4rUmOYXKTJZXUBQnBJlIbb0nNQXC/V5CSGbvst8YtfK4dyTwHrsZCK0tOWre10UU531DA/MA7FEu50UaCGEoZ51M0pZ2TkYpjqdwR33O9tociDotm6J14R+6DD3hCeV0NR96R1hxoOltoXHU1XfD+82b/5+RafnxhX8dPcsbOt89ahwWaxBE+Gr39CSJ2/aOILFs0lqRfHj7+Cb4MPV4Hhff08MR7Eteqw9czPFa7Qg+586vRk+6P62Sh02Lo5Vm6o6wPo+QLCAS/wgifc0YtALsVvERUo/FjRpUJUm2FRgItMU01DLUVqPeWOfz1SbbRNtXM0vyj5ZanId7ecMlz2Z8uSVWFxcg/hNP47XYEvNF6FSelb0cjZkglfij0LVouma/4teYMdxs2xygY0SMp50hlQKZu2xvAsd13mlibJrijZ7hhp9Mf0UqbaaZUl1Phmqx1VOhyBsVvUB2Wfpu+bP4BUGcMEfFIxCdkLWA9WhkeHrn4FlHU1Lse6tZaOUzI6pa6LmvKbC+OfrSYBeNuxdHeL7ED5DayNJ3SX5mCpMl8UUlLpLd33PJTJ2K/dizLVWdg4ndRWJl0xB8E7Q3tMHXYzm2cTGB/eQp7RfUzfDT/eSSgE5aQdW+IJ0oqmmjAUxtZMaAPYyqo9ox4loPSkU3H8BiEGBmdD07PO2UHz0qarsRw/2xJ7ouVKg2AsdBrGzwEd7SJrBGgdnXX4EEBI0J739Enr6/2KK3+KhX6jw+VgFYk9CatXHC1/ULPWWTbJ0hVPbJWQoBv746Uei/v6Ab3dysBoX333T0UYe1r6M7ZN6Z1POS1psaG1MyRTY2lWWBE+1zz/1NgClRoJZ2RB3L5DJE7yzNq3vx6OPWwA6IZlRvhBeOH6mnaDfGDeP4X5V07Vigk2Do8Q2zs/ZEOqM7BsdMacm/T9VHY3xuo+i6yWOUEj6YknHYbw7FJ+PFl3Nh32A7H/CMDX78xhoZw0+HZ/rzZibI6/B0YRbCAibVvhsJm/8MTmLrD5ym87SibNn6v06H8cDL8P80Yeq1bkGnB4hnb/UDckVlYTw/u62jk3x1IHig1ObUMov7s3gTwjlm6MYtOzqOPvCkT8kdOu8R34MrFzsZAfGi23biSRsrjB8XkSu/IPGjfp/2IYj8t+is2hZJbaZITRzl+c/7ElQX/pyJHoPe9Gypj9E2K3nPoeQFVEwuEobSTXJg9bkXNNnoFcv/mBd/C4bKJTmNK32ZIPAQPDkb8s+CK3z+8yVqsRQJ7kZLQSckxU0qqp6V0ZmOL697/ABsDx14=',
    'scripts/features_ci_sweep.py': 'eNq1W39z27jR/l+fAmXnJmRK0ZaT693posykOeeStnfJm6TvOx2dh6FFUEJMkTyCiuO6+u7vswvwpyg17Vw9Y5kCsYvFYvfZxQL+/e/Odro8u1bZmcw+ieKu2uTZo4njOC9kVO1KKXS1i+/m4nYTVaLaSLHKs0TFMltJkcpPMhV5wu1apnJVqTwTcS61qHIRoe+2SGUlxbMfnr15L8pdFkwmL/JSgLC8O2Q1E1MRy7SKhJtQr88R0YvvvvPFd9/g92v8nvvi23MRZbH447fiK4+fmN1kKyF87IutjDQkj82o0//9n0tRldFHSJeXCqJFmBVEIfarKr0TkRYqE282kZbisXCzPJvmZbRK5aSdU1LmW6FZnFjkuwozk9rHFItSgatIVKkrUUax2qFZq3U2jW7NQKn0RbnJp+s8j6HNvPD8CQmdNHpopLsTt6CQq7yMJ5OHA7WucozgQmeV/FxN9SavtEfKNzx0JQvf0mS77bUs6Z3Ru6pkGREPLdwiKiMoSpY8aeqdqCxKoUYdVf/wahZ/evHjO5EXldoqLUnOKN1ZFlFSybIzqnCJwDGd/8GduF07DbdtjlWF4uzKbGVWmemw7cCwSjm1gzG9L56H+C7cd3IViFevxF9qK4tLDD+fCDt+K5dYiDyTQmayXN+JP2A+QlbRFKJuRTNnaE0llVjTOkEG33RXbK7gORPbl5Eg8fOykvFUZZi7Wf5dmpK9bre7SmVrcMh3hSahXvo0Dka/aGZrZZBliRUeastng73dwFapPVptxGqTK7jALeyQycksxbXUlQ+ZassxdJv8Vmx3oLFjqEqk+a0kYydfi+jhVlWbASM7BJzvPZkUlAG7QlelxU7Lrila76OZYtolxoTdG3vSHgw7F7FKEgwIp9XgXd1KmU3Ye41jZXk11qVmpatSRlsNSV7vqmJXzcWHD/BFfZYYuNFnT1bww6dnKxU+MQ491elu/TSMn7BsT4OPOs/SDx+ES+udKnwU0GTrQ99PaPLk4GC3g5sa50WjJmNPpfYwvMDP65/ehD//7afw/cu3l89+eLeYWfwTelWqomqFCiGNxjSKoLgT0ylJKF4+DvWvO0w5lL+GGjA2K87Dly/EL78wb/qZTs0MtHBevZo+E3FURdBhlEgH73g6WpwH5zP6eEQfX+ODvl3g99EFOvWA6/G3aLnNyxvy3gsC6QlPLQyTHckZhkJtC9gubAXrYPx1MqnbyjUMRMv6O0mzSiOtpa6bSLX1s75rmuGY0oxURNUmVdf1MG/wteEPKykYTLNiMnl5+fYSTkEdXIgHrYehF0CVefpJul5ApppVEwwSEM/AuJoLcIeFuERtu4gz4ehy5Xjeqc54W4sBsmoWXpcqXkshfg+D/DWai8vH5xf+i8fns3oeZZSmiDiWCNYSVqWKUgJvmFufsKbZpQqwpNvZo+HSIBAcqEcihPsGgohXniEuJRytJtxEW5VWeaYijAqA0mPDdYKCJXv+8vKnV8+f/TV89vz5394+e/53H9Env07lNoyAZTSHKI4Am31utfTwkjAKCXbDWBJa5FoxuNULzqAXNqAXWpAbUwXFysdmtLCNUJbRT5fvX77+4Z1vQurjoTiXP1++/fHv4eXbt6/fwkRmcvqIurwk16iAElEZW+wEvHYwHTimthGSCRu3TIjg0DKZTGKZmHxiDUdwM0DcnKzDN4g2F0mas5I2uX325uynupArSGFlXhLhFb9IlEzhuQtxnwTUyiCZ0OAdxwlML5e4eHumU4lweEyHsRD9TZ85TRJRaQdYhH7RCNxVn/F5vYOpAgmz/CAjaqCklHDwjKWddL53RSllkUYrybLYaS/4k2e9wC+cZMK9xTtO6pgTKS7EqqsqDF2sZeILwjervVZfPmJkijwCstUaFNOn4mes0LwRk+gDhscFc+m/ADO047PfbKyJ0IIfXCL0hbOFoE5nVBbG65MCEiq5BWmHkW1sOhoPWxx6ndthEVhPMpMfjFJz4L/9V9ZHFse8x2WaAT8TmPtCc1u/2zb6HHZSt37//suBvLXKSOb6edKstvGnkDNIu+KFVryavK7tciLCPO/mm8c8Esm+yZ14j4CsAo6MZpdzOkou0zRfsaReQFGr5i8b8FwM0LS3NpR9hczSJUFbbVYIcikpZrd1G15BUkZrSjFBErv3xbyziMviiv24IL9c7z3+sqYvndX0hl5nhnn4UFwgGvXgi9paxbaQbdVaZmu4dRGY5Cf4kfRGIvoWqvD3Bggaq3LOoVL8k/0JM6I/fiNH5yeWUUxJj/XBPgWvYaxW/SV8nXE6FAjxf5QafvhQD4okyiamLYRr5IoSHXKFdY+RCXLmCDxR0TpD7FArhMgmmcOSeN+LqKMwtJEjiI/5tWBEujNZrdnVUJ7KDLXZNXSTfho7pix9CzCigczejjY8KtuBIlWfkMLWQ9m51PqghJDylIA+XM8jNG2nxVJps9e4zXcppdMw1VvKu0mOopSfVL7TZN20K4ht3q6qnr2aSZArLq+aRtKm1X+nI8283w+BoVY8TZWCQx886edGEhWsEn7lYglcmFBwDWxe17YTsNaXDv9xrpoHT3wl3Jl48gRpIjJ153x28dnxesytpI0UZyJxyGbDewy7p4SrQgpkEmynR2llB0phZbTrzQ9MkxyJk3H4EndFnh+HBB3I9nSRqorejpKyJ5d34y9abQZRUWBlXBIvgO0j7BJLzxulk59XsjAZbfDnd69//gFLF8tLyiuOD3QNoW+at4iUpK4l8vKryaR1wMQalnvDwOmbjVZY7yr1YIrQ3Y14gnieuWYiIyqwhlVP0fRb3mB1VZbkWNwDChJueX5FxkI4cEgSIKdwHQ3GWUxxFLMYURULt1h0pRNTMRtX0RFbpJjejo5OztXIBBlJD6SE+bIuZexc+Yevh4qsvb3rQIwSHdfH5v8iOBcPGxU9bUHzUK5IIVlB7ME2uPeSYk4dcsfDUK+7TR9A0abk4/kFccGnc9/kSvuQCgP3N3vHP2qZrV1z6F10WZsmlgkpNAeg1hgXQ+PsDVGj9cIoEFuEBDuEHaC/dPs9bc1rkIXY5D+kmoNbx7TBVGEQfVbRtV6DD+IintyT8lHIKG3fck05ljsl8n4vhCMYsSC4REdKBniIpzzS0koe1IZ2BQPBpmN24ZGpzPomBqujlN+5ceYC7t1a55zZD3l54yvmpNG1TJ15z3y4TR+Kc4QF1W3qUUkLsGSCdZormunPEcI1LYpj8wN3XAXEaR0ih+z1o6QSofMIX2SAZZUglWtoarYA+TJadjpcHeNhFttO7TrP0xM8gJlYpu9OiFPpA1FMWZT0BCuO9WDZTONRjhYroV3q5464BLDRuowvLuyG7wiGkyn1zbSFbHq3bMa7Ggu1x1ME+uEaH4fZHIO5TuR4VHnZAApTOQ7e5l1wW2LXYGJovNtig9LaOkPunIUjBQLH58cgf08bzAQZe7UIAdkqI1dyfskGCYdF/aH5+TxGG1IPgj+Xjyms1CWNLpT6Nvx2toaL/rbHF/3d0WJkO9UKajMFEwPmY/LfOwW/hD7elzu5n3zxWvES7TLEnhsXea5W2TrMbxbExRvugkgqX2QhV6vxtKL6hm9KISbl9Ntf1hCMKeDXrQVRGsZFeWWL8Mv5dHbVFyr7zJtPfkufQaMVwsO+NSKoDKKgCYDgAdMjefFgJO4v/TZsYkVvv3kQOalyT9mH5dh/2VFN61ewBpK6826Q2NQ6rEmyQRpMiq3fGfgYMkT2gFwaiQQeYNsPaTqDQcy6NENAem452DzeO63RwYBoqqx6zrPgZ3z8EjItBQxeFSyZ4XaIVE5JZwZsi4yfAwKAZt8XRsDOafZFYR9EKXKWS4usZqNckiFZZBuLDPWxjyzDjvos7Ha25WaVRuh5MUJZRRc9Mbh5lKAXQupRIHWn/QtFLzd5SEcrfT69CLRoC1c2afgy1o2GOZqYjljrvop6fu/URosXDQYcU5c5XpvXEOFYDzMmSZz5Yd8p+bBySmmPZRhOR4oFPYmYeYM74wjTVLOOw0zb5b+GNf8ZSvybMNDFKEKEcWjzvsj/G50cAYFWZ6eQ4N9zvf/E1U6Y69DkltoCENtH3zhGUlwnimMGseVIiqw7ufoIO7a1q31TzDaR273kOE41RmOHjuPQaefH/PqB5u0NH3cW34v6eLCtLPGhhqq6ta/6lCFDYko8uA5kjhlMxuOSOqQdC4mAQnCn8wt4F7/xadPCaZNadXb81iq4CznD1u6zjnGg85CyVzPocQDOK11ZHmYz+/6uMHUOt8KTFdOzsm9J8kHhnk8FaZdVnxAGz8r1jiqob/iNG0tzIArdLsIwzldh6HUoA6xlGFkS1zHnow7t5n/dqVLGnXTnCIE9JgVNRtu8hfMHp00wlweHp1cnmZlz1T4v0sTCHmM0fOnklesij/jz69NsEQOOMDoPZicpm1h8hP7gXO80t86JcM1QZR12F49P0tPB/yjdNyfJ7LHzKOXp6SP7r6moxt2S8eGw0zv9d7yTrBLsoMAr4tC6cDS0IMMK1oXGjUyLhUOnrLpcnVFPPisu7oTLFx7oMkBaAbrolSw957QJN0BwZLyjlRojRwdIBiV2vkTBhehI3KiU7xOVjE8xqcoWzvniwmkRsZ+ZbvJduyYDszp6hNARk64jmR0t3Wpps5ZOpbxTDeejnW2U3Qke93vISTha5Q2UnpY4z9K71i8fjtrSl0hNgxIve6eje1KexfZwIRe6oBI0n1fZi2pYCMwRn1FKata1DZA8VLkzUvMfklu30EzfAjKcFobtYXdtaO1utjE9RvM0tVya0qU5Kmi40raU1Unhid8gBMpBQfPRH8/PuabZJzGD/usiHt/hw3s+9nWZhzlk5Ucgm31qdw0TW/MzUUEH9HgmGsr6dbC9iVXpmqsamoGeyuEISf1tLjLPkHRCt7SYkzPISPnQwanVTZvpmmTkzKF5xcUMc8jQqWjwbINhyutR1SGGmIAra6UlZUqJc99Maz+3l6bMvbl7s1lruHG2trfVCWRx1a4Q9yerRPPgPNnDzroWnaQ7venohjIcvmCAjMMsr4mG817KbS5m1V1MjBvUGrj8Suc7vWsQ9ipAu9aDwnV3VRJnpcJ7ugDDpN4+jO/N/Yn1/sixkLEOcsZuEYTuOpAp08Ufvp5Fqv/cQl9z/4bcEDkWeSYvprnzyNctQHqAA18sbajuH0wfBB+BwC5dElLmqFdxJslH9W4jueeNTi4256r3++GUj1jl2EkYd/1NT8LossrwBOxod5rCkk8R5YqOVLAGSJ6u+MBm9dsdmxXIwvummOarmyN41NcxHaZFn6Rr7cMEaV8UC9LciIagftPHHG/ZutwRVZpMebx4WvwWlVOj0Xlt3Y7xXNrBs+PZe0Dz2gMfPjSy79tq6UG1kA9mB6fJXVdrE4tDYTv0jYdQ0xEfcY4ysLA+AuRNhfZGFbxZr1wyMu/LYOFgQGbzT8MHgWctjVd2c1yP4RTvW3/t12ibq3tuGm2v44hq1r5YLerjqHjRXrIwYN65p2H6opPfBGivDoYdIY5mJNyTUmpf2Ax5wU32i88zXNCHD0AMzfovyOQPVEZa+hfQUqdZfQXkt1weHqLClwMRjU6PAZBSFa7XL8uYmz/2DA5bUXdZLg9KiN2SG+QZHFirQwadoshpWvnpgHa88HGazUiwF/d8TfDiXO85Zor72fm5KfnYf0Pw5l8Hs2T/lRDZgpMBYuzNH8V7IZwRk0hM7VQgl1dRBv5FYB5dcy41n82CRxi4eyP/HspBL/Sh0S6So6ypGkfzBQl0UpN8QwIS1HFV2A6JN9DTA9v44EAz81PjuEdSGgb1JqHppTGTiaJLhKTOMKTrBE4YUqUhDB1jxKbsMPl/bT6RqQ==',
    'scripts/features_replay.py': 'eNq9Wmlv20Ya/q5fMWU/hHRl+kiKxWqjAN7UqQNsDrjpAoVWGI/JkcSaIpkZ0rYq6L/v885weEiynOyHNRCbnOO9b+bHH04qrU5uk+xEZvesWJWLPHs58DzvWhapWDElo1zFMmZLKXSl8HDxy8XnL0xVmWZlzvLbUiQZKxcyUSwvymSZaFEmecaiXJfMz3KWyQemF3mpg3AwuNgDiPmfF0JL9iqo0WmCx+Yyk0qUuWLRItcyY6Jk8l6qFdOlLELGvizkoBBKLGUpFa4e9/AnmslHEZVMZDGL6cgyyRJdJtGQ6ZyOA3WWZHODTGS49hd7SMoFvQ8avoE7iaTG+ULlcUWPdJ7INuDT1T/YQirJHgAkNYDpJntYgNyaLxYnMcvycjSgq/989+tvYESklSFUs3zW4cuQS8eWeSxTENBjqhbeUmalFfBbjv2BTxc0BGFvMaHZzc1MihJnNY8Srh8kRFasbm4CCO5SRAtiCAoGAmIFwooWMrrDq5hDoYDck0KeSeYbBKm4lakeWmxZtbyF6MEAke5WZ0kmUkbam6/IRs7k8d/YlSD1f6rKoipHoI4M6MSRePI6gqDenIDS19DmIo/f8Pg0PP05/FPnWXpzw6yNsVmulpBrPhs8xR/zb248nVcqkt6IeZZLD3wPWZFWJBejNq7kDFrLImlQ3NyAOIafTx8+84+/f+Bfrq4vL375bXxW+wTTkUqKsiWZW8jAyY6PiXp2df6Jy6/86h25z2Cm8iXjfFbRac5ZsixyRcYIO7B6HwzcmprDjLV075G+d49Em3vWK+0eYRHSYojyNJWRtaN6M5YzUaVlnESlPVOIcpEmt27/M14b3NAgOIC9ZMVgcHV5fcnG5oAP0pMUhAchmM3Te+kHIaiE4Q1ASEgwQ9iJVKV/Cr2Xyqfb9RF2wjytIi8IDh3GriMD18ozfquSeC4Z+xHO8lWM2OWr0/Phu1enZ44PJcBt6hjRaTXvH3bnqjSBX+iWYyxcWo9DONlzRckKGqyPL8QyScs8S0TGKWzofTdKJf6E4HO4bX3t7dXlh/dvL/7FL96+/f364u0fQ3YRi6L8bQXnWA7Jz7ighS1wjuhCKi44eTWP4XcAqhMbyWp28UvG3MoR63yu8qrYJm5w+fHy+tc/+OX19adrKBPO93JwffUJj6fh2WAwiFKhNbPhXaqRsXqYDGwV4bHk3NcynQ0ZmfSIVBWw4zfsIwKAPUo/dCI0Nj825/ob2vCLrQ734W2VpLFPh4P+aSvg8a7Q/Q6wELH3NiUh7oFQi2H8lID8Fs/WTa7hilKPGPnKpKyKVA6Z+TMFuLUfjJDAivAvqXLtnyKCnAYbEjeZFoTim+SDqNcN5iVlAIqvSbmNzUZEbnJhH+cszUVpcG4GjUIyhPJaGUA4soQZbZinCSjLYqGUWA0RHstpq59kZkhEpKG42WO1OUM/1lmHjJPw6JRBiauT0fHZNNg+a3htztaaqdOT9FPkVroLMYEyURQyi32HAbE86MPrUjXBNeK+ESjY6akfJlvyjpQ7sJREgM12wbVy7Ip9vzyN+A+Lr6e7w4JxQuzzK5vwM96KRz1OZ1WaWjZ6y3alI2KLNdgSaonckhId1dJvEIYzJeZUMQBu7K+LUcfxJsWUcioriNP5JjAv84Zt60D7FNeVh1OfxX50xM4R/3thiNb2a2wHTqs3m2BrjeF3neZGjMQwIfeZGu3RU6uSVrVkUC3pHeuBgXKntYjqJ3iwUrmiGDKZDjv/Bt+hZAvCGX5XeZbHPfokJE6R7LjnVjMUNfZei4F0o2vdOGH0VCMUhV3jPBPPHpKxt+XIsO6+a1E9N8HVKfsBzLUXudnypn0cRoMCDs/+DYHKS+Lan3nmLEMgQHmG4Icaba03I/zaj2rj9YnKTQCn3z8xHweG+4OPvP8mJyPJOEXI+/4Rp3q3n8rMwOmfMmbRgmBHzD8Daed4aC7g2ZDSiy+7oP4Hu9jj599gHrVfrb2OFFAC9wzfc+xjo3UCz7DLTQ2PjdonvJoxywBBMg+dDLVVSRtO9rikbcPGbQHUZRkJN0/R5+H+eKeC6mVQ8s51a9YIY6F7AbFbVjvaZ+Tthemm9SZDnsl4rb8rZ2dtENoj6KSkBrWWM9mFAUW6OgNFpg2y0iPhGSxAEpqV4Y5TeTaTotjgfQUaf0ZAV5OeZqdB0GhOluIcJ00ec2e7OqWzuwj7aLbB7zEACNA8b8luuge2iGOjo8lhNezVAixsYBOA7T459Yp+U5Ciks7RS5s2hRykhBaatID9aWBND13Y2jaTiEOdcv1OrlDXdRZU/jA0hNqgSu96GmxgIqbQR9vp+nhqiXRI7Z2xgypzdZwhy6QkGE6nAfPpl7WcWZ7GkuyKqEeCNPUz+qSCYL/yBi7CU7NkgrypZn17LZyn+S3C7Jpubbi9w494w0UCutA3ouPqlOm2DDfdV4bGfDJix2Sl3p5rremXYk7Gj7sTOryN0wtG026lRFWSyFa+qSDoblNOOM0E/QQS5VmZZFWvaeCG6bGT0QkDVqJgw9tUZ+jcxuzuhvIR2tf+c7hgD+JpZLRrxgE9PM2dJ5DQPsDRvRAOGGu/vYFWIEY781j6wU4WFrfatx5r8M5l6XtqkXtDhnbNRHz6y95QC3d2PqLGw0yeFvmxlhn1hvdJuaqtMMeWakPpjkPuk7qbG/QtltyoX3PAH0ifkH/4C05cgydpqlbLY47E5gdbQmnBT3zcn3jWEymsUDwzS60LUoCauhSJvW/B/zRuODjVft+Gdpvqpo/0OywAIiDg9zgVy9tYMGXDMmJmkwN2gJFiJj0iphOYGnxkM1rT+ouWkheoiKY10abYtaDqTEOQ6pi4FAkY7rfkZnhEkcUNksILNa+o4v9sdvxY2ukVqBxzHucR50HnZohgzUV9xffsOAtmqOTXKlEyHn9RlTx4wbk6LmVY12PvJ283Kbif2trGE+/9e37BYwFv4VrMCCctnTYvH87bxzPEgq/tuVnyiMzgTZ9Gs5BpMfZ0dYvgnGRzM2fdDuYUseqe3cwJD3IpH6O0imWPyQ4zZoZEpFVZQnNKV3g/Ac0aRUdkRx1opNxhzYEljeVZuiL6Qbu9yiim68MkL8UjjbkJS7kq5DihZvwbkEA4Spfso53zQ3CiRnoYnb5LiuMYYIFPGN+BAmDfEhmnosVaJzhWw9PsgYb7LDeDYasTG2YPY+oyZQsBx5WZXnpmPzgIAiifg9BMex0o0pRJqQai+UMwdd1oRmke3VEnnCxlWEg141FeZXAN3/lzgV03e/PpZthOtECQdWId0uMJa/bddri8ixNVT1W08cuhFRfP7zpuipLcZTkLyds39LZZrs6k7sqeJNdsPShEO5vOTLaLq2WhfTAVbncCAcXbGESOz5wSFAVMxL+GqQ3V6OjUUZztigvZzwhzFJ7ONqTpWVrpRZfFymSufo3YwB5a2dFi/dhUI+b2Q07mTSPR06bwsgY5ZN3iqFOIEawQ/C81cdfJBXf3KCfv7ienU2rfVKnpI5LvvaMA5dFR2urmKMr9qJgeCXjtyoTfvBta60DzXDFDcOi486TEzqzIrc1XJBcm7Byre/QZwD3TmXnoNdc0a/ft7WDT/Tbj7ZBD3s1jR8T3VGdpkkk7hGmWnvMog5iU0ubtXnFbKyxA7WsFIB6NqUxb6dUrDK2JPAijV5SQeT8OoXi5GnbSdmB6mQxBRlEnb65usW16APhM3Vpu9bTdwc3etnZ3QLPpTYa2xj32Y9zYtHe9dm5KzWjvqHMKSMQ3z0NboxoIx3V3aQqabms7DXZLW2ocSBo0UWrqrk63PA2+aahUt28btoaUES7qJLU26Efh32cb5q8dro39Bhk0HzCbj5fekyVCKyVv3bL3osPeC5BaIzJbLRMo2WqMW7Ms9w3bjCvuXAPfrxQ7mm0PdCZ22D+g8KdrHg/VP6G0qd3TC5jzTKRpOxyYdBanwQFINqTfSl1aIjrvqFTHzPtiU7pnBl9bCEp9GLiCE8e6Zb1+nwZmPNm+W6c83bLwvZ3Us1ZqO6mfdy3PpSZ4gFD0PwJGfYODCdWftM3wI05mSHO6nQ7UX/xvV09aGszrTL6kKeIumVvWBnN7BXNbXok9Sa8XKF2/tPZMLDLCNDHJA/F4M5HJ9R6jJr15sUxLgQWK4aS+7W/lwwPu4jX5tj/delLqB4HtzM/Ij3cmZ+btGaJEtDCeZMX7es+32EP3m06Pb9lytfQfG3tuk3Tt4cFh7v7f87ynfNfhe9zy4e9kRy1yPs/zuA+xE0zY6zFNLNhPdmDxvfDbdpuGkfbKkB0dqU3QK0y6VajnhX/maIg71ehjAAK8/2Rei994SydJ7SlFmZsUvjzVeCFbtLdGL5F+THnQxAL/+Xp1uBMJgI2AUt6vv1IiYcX035GyqGT1yh6Pd7SmoBVqswZ+XP/fGhuJqOB2qfFe16JDCDMpfBSegzsEk2AwGCT0lZ+6Rs5NBOecxgmcezYm2tnC4L9DGPnS',
    'scripts/features_report.py': 'eNrtXVuT2zaWfu9fgaE7DulQtC62YyvRVDm+xKnNjFO2M7VT6h4WJUJquimSIam+pKO8bdU+72/Zf7Dv+yP2l+y5gCR40aU79tQ+rBPLFAEcHAAHB+ecD4Du/enhOksfzoLooYwuRHKdn8XR6MgwjA/eLJSZ8CJfLILlOoXneCHyMwl/UynFQno5vBVZvvYDSDRT6fmZSNdR9lClZba4TINcNt4+TGUSp7nlHB1RJWIgxDL1gEqUi9+WbvCb8JZeEGU5VScjmS6vRRhfyjSIlsiF9OZnYokJXh6nwpRX3hzpMbmhEPM4WgS+jOZShPJChlBFT/gyzL2S9POXz3/6IIA7oBHEEbc0TvJgFWT0Blomk6wgOhIik6GcU4pM0ziFJkdx1JtJIDY/i4O5zKwa4zVi8zjLocEC/nAni2yeBkledYvL3eIk19j9R4s0XgnXXawx0XVFsMJU4DKKc+b46Ei9+5jFUfGcXWdcNPHyszCYFeV+gq9lgWi9Sq6Fl4koKV4l0Hx4Af8nfvFu5eVJGOdA5eioenbWmTSN58ulYbUzAvf4RHTCXIh7Iop/8cbi1aP+8OgImHOQLwf6SKa52behk1MTeTOhqUEIDbUc6Is4vJCmBXlTkAnLOnr39u0HMRF7MmbTwenRu5//+h6yUomHwkDJM/Ch6Gbj6O3PSIvyYQbqdePo6J5I5UKmJDSJF8o8l8L0vdy7CH4V2XkQhrYIg+UZNDf2pTUWcy+XyzgN5l4oMmh0JoII5sqVBDlKfZnaYhaupUi9VSIWIKb0EhJJIrOj96/e/fAKWZ0a94be10/9J4YtjHty9uTp6BE9Dmbe4muP3/reoN/nx6dfz7xHxunRdz/+/Mp99/wvPzGNQf/R8OmAC84fz70ZPWqUHz9+9lR+TY9Pn8yeyAU9zr72R4snQO6Hv/6LLeBjaIvv3/3w0hbvf373+vmLV0DduNef4X9MZvh48EgyK4/kyGcyizn8NzOO3v/9/YdXf3Hfvnv56h3x9eaRGDjEOTwN+enH4I0YqZfDt0L+UjyBNAAnL56/f+X++Py7Vz8ChRso52a/rGGEXfmLm8G0HiR9981rYyxqxItMQEPm8zPKOdRzUuU0A6s/yIr7blRmq3OGFariOptlFVUaMr45+vDqXxXHxNZYpMabY/fRMX77/eT5c60XqrRhlVbUjonqmZMKrgs+VOnhcbPztARm6ad3rz58+Hudq4op8V//VuOpYkmlVBwVj5Sg9WLF0jaO6gwdHflyAc/XIUxc0fuz+GscyTErxjB30vlPHsyYzFknMPekeVNWZSziKHey4FcJNJ9BDd6VzBzpL+U8DmOsiIWX34feTIZaAry/yoP5uVPLXBG/bicqSss08OHVh3Qt4RV+K7PxPOF3YRDJy8DPz+B933mikSYqWQLpmZPHCaS/9sJM2vWUFFWLlsaLrrPw5mX71IRsku7KIozMu5BAY0tqKGH59J0FdLWMo7LainJITOmNAqGw1SAsvFUQXuPQvpQfvb+txXsPtOzGKgYXqjahbltEQH6MA98YaVSH8ipHhWkaSbREgUlAk1jjkgPkXbXBRI39UCyMG6S3cW6g6AaK+EkwGfaBq9ksvnKDaH4ms4mRU0dapUTNwzgjdpC9e6L36f4IZb5wq3P8MmCZTnznJawcr7F7uUkZLNpgCE0wCS0ld55dmMUaxCaOW5g4Lmd2IItqRxpfYtnpadl7sJTmcoUdqGvcqvt8yM5kpoqaKjGZqLKnZd5gIXxHrpL8elxTj2BF5UG0luXLLOEG5LELJgTwOjd9kF/ppSsvcgeJrSyjiTGPZTqXhuX4aZxEnmmVNGAGABHfQbvJBWMvc+nJfwXlHQ/0QiJNFBgHPhzqEehSYNrIYQIaFStnoOu7uMH3riLexVFJ4RJe51spUKqbnsXuMo793ZQW6zB0d/cNZjmkd5jUIW07lCKKjuMlCUx3TZnSHGc5gGnMD6g0SFTgTSgj07fsRv6iNSvpBx5qjSxx+Nm0bD09KBIDTKlTAQGoj3syR70H5o14gNIBFL12KW47l8UMSB9fFbkbOQItQ7CNWmOMFVl62yTbzBpoObvo6+NedZcSlLLPcOZhT6v3lgC7UIJR7kRetKP9iqhGUuuJOk1M2EJ2wzKipkCprkwUGJWE4od6ivWvQRru/cAtnDX3InNZdZGqskEb+fJqQksJU6BVBBVXapzM5DKIbojGZvqns3x2CiXg/Rwokdo7yVZeGDZNNMzhJejw3HxfOImFl9V0DdkRBMv1w9ve6Htb/PzixfuXIonj0HLEa1xzLmR6zXpRmG9e2+LFD5ABPT9OMZp1c1bl+BJ15TjmqfcRvME4vba08qVPam9zXT0Bi2AvQRtHQrPBl7iQTrve4xOUtewY1mwlSuBHROegktNUhuxSArVjcpePxWWQn1GNVVewjBFv1K/opkJdKWXjeW51VPwB1DNQBZpBJnCGqmmEtak17PIM3BgiU7ngkBp66RIrKXgCKYiZn0zvjBU6wu163xeVkD9dNGNc6ztqINfSYx7Z77bZd+cMrNY5gfoFlkjqzUnfGRy3K67T5baDaRFcgPvgi9k1pRJTSZxlAS74eifDjKr6F6qapd78XObYt69hBoJP2YuT3Bm3K6ZxACngwQMKopSKTBULVmDt+sJb5Grcqg6H0WGlDukmWe22AAvdcjYdE4iMYZx64/c0edWs3fAEXMRxHsUwslDZSSZz0B7L/OzmBPKD4ZjJZHMzSvIuuuWUXkP3bW7ClP8ouqCs0nUo2+XEffjvZLUOweaOw/Uqunm0uZlvbtozY9PMOKSMC61nN+LkpEtnrAIfazfD1LoZ9Z5s6m++7j3taNB7NpHuF5J+XxTzEB7z5szAfCi1tpZUvmIxpOLHPVwz2gJQo96ihK3Chiiujcr0S1HOcgdDV/k6AcEzdbMZu2YiTDCWU6drGRo7wwV2KyZ3rSmU3uCVbUSQEVxEggyWEbObuFpqDLCN74OlrdlI7E8oOwSYA1d5miqb9LTgh3q9+FI3Nphts5YQ8FsLCrQZ5t7AXugyOMZOf7E5Ofmi3RFlH3BldaPioPq2GhdNsm2joqRPIo6iDWKgL6VfqbU0zvN4xbMLhQQ6tZyGtTew1irJOWARz+UVWJAUsnXhOTeNk8hwPsZg4FDtlvhK4Cvlk8h8nYIgKqeP/dVB06cHjXWo11MqN93zwfJTA9d74xTo9PC7478auKs3HjtWJEJkZGTsGTWdIpRdegm6zaTyXMZSUwodVXSkkU3wF7P1DEOYmTmwyZBSFYAtCDlRR07MofMEbNV64tD5Gj5pYK8nGCywygnrXdmliU2uODLza5CYMJs8ULfg9bgD30QerCJnBlYMBQituk9H/aG3onLoUG44xZtl7lL8WQxk70n1FjuxeFk5f96Vk829HJQJeBhUEFY/ygusTJ4pjif0Cf0UJmcerKSjx7YogwPZpF9N9Hti5aXntkhAhdOEtmvrNug2PXKPA88GVLHM9srVVcs2T2Ht1bwkeOfaPKjUIZ39ceos03idzK7NKdtxGGU4N06hfeQkKUO17vOye5qB5zmfqo50Av9q5V2ZWqfhH1oCqqzYY1tygvyhX6P1/7ilPrRhwLxqIIpiOBSjJyCBRTAHXL4I5hi0qAyAZRMKc2mjMsCIza8Udp6MrCZLyL/OE40CvdTFZyenVW67Ike8QsUoBlCxcWUUMnQgf1fQqzAzwniZJdBes/fIFpDxsSZkwAPOURMm1kCOYC5eiQcPxBC0ylCrDJ7DywlOY+iu2mqEbZC5ewUNAYPAgKoMyy7eXutvW0XCYGVCxwBPA61I9XYkh61CeQCz3ORQ7FQJJwxmHOWkT55pPHdUiMabafy2PHaD49/A2PMsxVaH+pj2TxVDXKrhgdgd3gdYsRXJMxCCkF021IU/wmANX5pT4Bb/FkMaG9yn8M9qMS9FcSXnaoyx8olR2NQ45z3w2lgfGFYzCC92VHVVVaULUb2CUmXcthYlGJ0yw1Usj/8xPH44BBW0Ti8Y8RwI6q9y9XA4mGqqvpuof4FEDH2zTlATspMLFUVQ02SkQpbgXnsRuCrpxOw7oFFhQjy1NMF4yoNSxVNVVLhz+TY+W3wT1g9VWw96O1ihS6w5ohYbAGEM63viBWm2LQa6Au3gIv6nwW9lOzSCTIWMgKNi3dMNiJIOBTvKb468CrIcaycjtBbUUNK9OLx692xRcQDVqLJlJWNc6t68LlyFFJTwKlnn4I6VzvivMo17F14K5iv4osu1l/q1tTxBJCECXWpO/anP5q/400QYb16DsaPiOL5qjm/X+kCxY4EQB8soBmuSgy+V5XEPfIEU2EhLEwktu1yCyRYjewF47ZcIWOJ8N98M347Frt4Yxg+wPyw9fp96ZFbFKTSbzDpnGcYz0ziAjmE1DBu9M+pNpYp2NNSfGqUNSWA+WYpsw7jzM/TNo6VM0WKEIVfvQW36/E43Zv0jPZavs79Npglrz+ZBck3jlxV4e+GlpEelCOsT5I5RfczDGxoIOfnhhx5BlfDvc6NhyCBlf4pR4y4z0Xd88PeXEb4mgm2TJeuCBDphAfwTn6MtNCX/cBFE4EKYMIOh/1ywDxrE2caKz6fxuaONDwW4gJ9hPTdQ49w6bShZH/NGFduj3lsi39wdFPmG3tBD4dlhZqQTUbbMbtdGY66C6vE5BpXXIMLQEumrt+Ba9crkNoG0CiFjl6le1UPvaISlbvKs38gCf03L+WXtwZCBCdJ3nnXR59JguXaWJou2o9T8LKjzdRboLFFyoKcFWkJZFyZsqQDdbuw7V2s/WsRdHVDmrXFFuXW2OkShCCekuJQ2FMm4msewAEdlzegLRk0JJAUAi0Mwr1dzx0C7tsbranRnzF0pfox4Y/AyiMpNW3rg2JzFsEKxxCsOaPr6tSnmV9MXNEY9pXPuoSdC+qxHhn2v7wzpsw+WTfE5JCOH8yOYq8CB4/RbzH5MsYtjfDwBw4Ne9h+rt7+lv32rfcVHzlQVTE+Wkr5xBdT+mn4NY1ucBYxLF044sT0d9wan7Exl08H4lLNkmk6dsRIytfkh/oxKnfSp/vZbqIIshgAeB7x619InkGTV4NfZQfArtqdUasaM5hayaddVzKycZMU0mO2ZBDwTll7igozoxdSrvUXr8qBTqKfsJcSBMaaylG4/eVZCg2aTGHY/KjQFfBXTbNacZ9htRVrnRJvVJhj18ra59WkBrRd1NIci/AXGVVja2R60C8SkMDGzcm9kN47VBYx9IzpgMB3oopKWI56HYckc8AR+UMGhz0bQdoCDBtWn0PQ1sLtarXPkH+UbbDswPtGOGautc4vUW66wjm8EmjWwSKTePEQFJheLYE59kyUhWLIYOrLb1ZL5DQ4pQirxXHUuAl8IseCOTTTLq6YwMEghKjIJxfGJDMNjYdYQIWsXSFjZMOL44/FYHKeEETTVboEilgOrADysD/EyqAC/fkT8Cb90gIIwjScnUPgsXd38zUs35m8nZ+DzKhq94tvH36yHpp7vxNbyfdWZ8tE65khdq9qFhzgnQmGXZ8GcXRs1pLg9UkDzoe3qNW6D5IaCf4EuEchmhmsOMf/tALpnfiY98IlBphBsRM3VgY6pwdD306brCGZAb504ogMe1XraroBO0os29XSqwaRqKrVrbaCN2sjWEEHosQI3ZD7NdlxlHxRH68FnAuJuhcQ1ILYRQWyMR0LnbYHgcjV0lLwXgHt0WwBOeSf3mQcE4lJEydAIcG+e9U++2NBXlCmYqikKoFXl4tclvmamNoj5S9r+/co67ua3jbPtc8ZwHcin+Y6NVUgFrA2K90vax+ORX6RDdw23rQsnU2TRUAiwmj7bFF9+qaAy7iz1pRw0eC7s5vFXCvTbAluV0FXpRmgoYWVJVPiVsu3pRa9XvPCu9uFjNUyuZtqXLOqJzar3091lzasqtmFqhQAw6qoJ/DOYUJubE7DQzm6UZU36pQiroK5WxrSN6kHTRJsudBrog41BE6+Q7vu61DZnG0xotMFOBFhjvBptKLsSaPfjw/KRly6osokWz7bAxU1xSx1gbUP86LJ0v1sUcOrrzAK3mxvgFz4gd92epBKbzR7stNOQHO8ctz+Khe71s+4Gi8JiOqtDo+1wUgWTdsaH/D/uk90SHfX/MDQ6coYd0OizOjQKX35ZSwn5tb1idaS08M6w9mn/tIRFrY5gmIaqmmVQjNFTKApVmxwhK18OTq1PGi3TILFMixFkTe9nHkKLlEc8wi6ZDIZbANbHDYDV3qb4GKeAyUOMbZQmMqwuVHOaHRBvayCRVXPw294GjR7tQym5oVubo8ODgxoa5l1dYCqeGjoM4vOuzqjAYFuB8RZ8D1r0hMIVTx5/FixP9wrSll9wAnqFrEpbsz8LgE7Nh704n2a5PlRG6l0hPm5POZ9KaAwnmjAbXpt1O3hvWzWDWjXPhdnt/t25sp2wJZv0F1nlcXwutO/ZIWjfvgXqMwB/2aWUCXl3GrZHLxW4R5bEVA+z2KL+zQ/m+SkrWdwra/NxShsP2XFQjsfmZlMq/7mXSdcP0m5M6QGettLVdqGpRXVczFnCGlaQcTA0ZulRtmJlybSFd2uwDfgswMKSMdy0hKERtzwp6ODZS0OvpChXAYV1YAKaXxr0Ezq66WDvgpNSFCTgiwwNq77lhdipOqdsKHfQPHAfEDuh0VzdaKWEOYGFm1U4NJvIkGnyqu37w3Q8kQHqviPTVlSIWlxvJlKyWpkyibsW06lRHrLNjNNWLhSkKgDaBnHA+T6LEVMBQur5lLAdsI75LT+e2tvdCIY2yvPDVfgRjxBXFDDyROFBpsuPe+lWx41Vser7Kc33yAtdOtOByCTWTbtbIKeWtL+aFIOAUnVE8WVvKYzVFSplLBD7adaLPaDJbyPVwnC37D3aXY02DyNWBuTM3orRQkZcijdxM5sv91Ip9lenrrzwwrU+LN1JOEJzF9I4Ez26MveGt+C4iNHDC5SgCOPNsPCjEK9XJs60bKp0DL0/rYx2KrKnHiBQ7ifVamLJLRP2sosBdezGNF9gAHssFjB3c9yuSKH2KfBYpdZZBMuahhjmM40r2Iab9nSnEmAVXsASWVj6GRrywGajy+EN7h3AM6EgYt26h0/qfxLFcIdZzX12jvVNGRjWR3DcHFFIRsubE+jpsAr08agPAJ1S4onACTkLa12usbOrrhyLaFNzWGuQCSpaq76im9TJFi/gn2mn0VDbfTEkJsYNqwIrH5NtgXYbDiQdL8Uz6mQnoskCpoa3kMaWDRvoVCPlKS0mTAO9IH46vePGDPYf/d2HLLODUD5oIi6Fmp6tGQEkqLZYaoZAtUNBiTGM0jm42qG3mvmeOL8Yw1/0hMd/bI9EtSrWzuyZ5eUaBMQXB6OC8iDhshN3r+vrCjNcOo2kXeh9tYAWB+KWTvWuOsWn56OtB/VsWzYg8LStL9py0Vi2jw5bWSr+OpM7Tz1qTGxdrZCfbevVoawVmx128wbm7yepLfP3Vpjlvun78WIy2Fdrl8HKMocbmLcfqGQzj5bwihn63mjt7mVeJdeaxFQ6mqCTuhWfNbuQJoMyD5dOw2TcNVdYkLbamMjhHiuzMNFq0x+P6ZeG4z4xpiM4LeWxdApDyEEzCDtM0wIZv+zcGFWcnekkWdlCh5O942adoaspR/DMlbSSlN7ycOyDO28maNx5pOJK1bVFOpiK9xEVOXRmMQIZEzi/Ag8/74CDYYSzNYK/vD8AL9ghBP/5uECFC2AfDLwQCEeKcCi9NILkMnzDeCyd7egt+NU3HVsXgmXU8y5xxwEG+YFBkaQBrC9iEeAZO4yNrKGu6pDnN537HHhTw3evv3+vna7kDcyd51Fh5RkLdYq26kPVtl6JdHvYA454j9YR5ofcraulyr0E9QOirUrfgt44EajfClLELk+tnhf5vWp7SKUESy5420AAfPH+mDyu7eRoVWcenySrY9zhEfleiiDWRaDtl1CYe/aNEpg1VItbTfAIcAGk0+ENbJbaGhDA+FbnuNpV4ih6vi99NP4lFoMP3lahjuwGGQNo89z6Rhy/cIvNCjBgm+OxapkSXLyIieA4bTCFkk+MRrKUQzNB506Gxzb3JDYAumbwO557wKcWl1qvORXQdlyKA6nJYhMBaUsQANzpcoyZyvPSNDUuz2JQ7fUSv6xhhMttBPoYmerEMogxsveNUFcNQSP5ah4YrBDKQsmRM8QsvBebepBNWJwgOO3K6a3Ov7+iaujU0kP9YKp+wLvSjSq7muPx5ZjFoYx7Cb7Up3g9R0HwRakbqOnA5LPHJ18Ic/SEjoMtZQws4on5Qn5wD7o+dtoueuqV8JqFL6Z97Hxxloik5ONi83ANlmm6b59FgNr4FlssnMe3Ou18yD6Lcl8DrhIO/Ev9c1/tl7nPDiz8G1cKADORrN9vzYIyoWOHUCGrfKoHSSu5hEdZScD92tHk7WeN//geiE/h9+zYFLEp24UHjouPG7Rmvqw08JfqiDG+7TQkOcNWZHrggGVxX6h/biqL6UvNYvrytEDFK0bgQ8Ota5g82B/gvrnIg8zMpnMFsziDtd+HNbHYYN6N4Hf0jwLrawTrB551J416xs3mAZ7p7nTKrEN2XjQ8sbEzUHtF6q4Xn25One0+EnF6QIU7XBkahxs3WZnbshUG+0FN09wEnXLd9LfU5pVuW/3Q/SuagV0fsbr5XE/TreAypSF2NRHRproWj/3TpDbJp73BqTrOr+igy4JkKAXB9tquGcNo7StJ4sT8Yzs2HnRu2dhpbX+CU+xDM68Hnhq7NW65qyI/fFfFI20rxWCEKOHI6SsYKPEiXP6gUnNb9MQmJmEsTND9ldHKywwuwbDuFFfDdICnZiueUhGcWeWGYFyrzOqykk5CuwIN9t7IAFY4t0SZq2buPmSF31Wr7tDbLeccqfpW3eIgC/HFsatW1pL2aW1PignGgS0ysK0I/rf03Sm2GpjGzpTOA/+6iHQc8K/jI4gGoASxsB14DmvrgrxztWkdDgeu6dK1VnytxcU1bSyB/KfQeZnTDJnV8qM22UWhnjlNec+Kj5SjxMl+SXMzq61djDX4rKZQUps7WUgPzzzel6M32RbX8D8kT+BvfYvAKps8buzKAR+b5uSQXFq1Q2Uy2B63560E9V0iNSy66OBtYdDWlpytB9PbnapvfXEO2y2j14I7YNQGhcedPO8Wito+mZbeeKi7O8bejTOdjavI0+ftd9+0Lof+n3//D/Hf/ynML6z6jqAgAscKSnpXQbEpT23AKbaBVLs31A4QAnm27OTYtW59tuPbIw1UGXWCKhnHLj410lKhKAy84M2g6xXMtGuzUI6EW6iNN/CNatX0oVqcuxA9KokRUKZgEAnEG5jUwTBEEXh0E9wRt+YIfRmNLPEDPZjYyKpFGbcGP+8MUwRRbrahis+OVfw/IHBnQODTxOq1kPpdgu2FGzH3GqVVwvaSd4DlNkcHhwtaxg1pmTvHBbICyyW11r13txPmrZ++pG0XXTGKjO738TIMQZuzNcZtJxHdahqs1iuwKQpFgbtfrdaWKy5RYLUVSMvvjY7NDDok29SVxhyWomCBEVbeEolHdDgkZhmH7CCA0R2JGA8npez3MR/YKyOeF3gZP2ocTgA1U6hkLXxB2z33drvW63SRb5Z7aZ5h2M80XiMJw+rm+L74vVkWOkOVXEcB9OvKaA8fsdU1hPeaRwGT8uQG+DAB+F1jIKCMkiBK1jBeFJQOcnHp0RZQy64uqFr7S0nHH+M1v2pUVl21ibXmKiwOD3RNj3j2+IsiKksRQJqOeCBuRb8i4M3n69SbX9cNHuQdl13ilgaOW9vZRZzJcrzourgIRd1tAH3euFeBR5L+nW4nSfU3776ifsAzC7zFgO/5RtHBJ95SaFrTwamT8qZBwzYsS/s5iK9Hg/pMo7szJsWOBtPkGkzs/MhuKS6L5lekphbzXk4vtcHHasxIOuI+nfHMtPH+MNoPqW7tWMA3Pk/8uLEd3sdwfS75+HefgFsgpeYLlMVoyLTfvBssjC93VzcT35a064XPce/thCIHRGfaG41PxVdiWmZvuWt07Bvz4iI3ausVJOl4vq8I9k87lJWtzpK7JZudu0FnmIz0urdiLTslCpUDUj/tLLNT662jSu+xyLJoHKjyVGQN45tuEnoBHo/fcEDkISxojjBvSMA3oEPr2u62EPCIV/asadi79NMv/xwM+H3zJ2oOgXzpGtsmpNt1VtzehpRtAVXxeG9xIn0GfXzOaFFjHSNFCNnIW/n9BAZAh23srYeum14cH81TPkV1dW+0Xs1kWke3UIQJMWUM7hLvM8ZVpLrkmUCSVp3NnxxCvDNTyKjHl8JCs7+j8OYYFg2hC28VjuMLf5U0a/yVUbo2Bl6WNbWljI7dV4f/q/P4iPULWgkYG14ECzwTv9DWMMtWa9PKO4eyBFB3VMxi5GBspceGDvZTDT0knFrFFg/Ewb+pYZ4wGN13B/AqDdMOOnUbNN4Bg+tHtWvYagPublXawL9RMKHmNgZ9GOzczUXXVoNbI8d6q5qA8YvnTTok461at5geBT28J4wjNmguIo2o3pkNYBqtKDB55ngfdgE6d15DUUevrQJIpkXgIDS5mvj2LXDkPSAxyPktUeLbHcS/BURczfUSJi5/Z+x+c/ptxY5bmPGt0eIXz//P48JzD+3ia5kZ6uRN87zCt2LgPJE95d4YUWzcDlNmWVdYbl/97QaX6xDx9jPOB2HHN3NvswM43oEEo0Tz/d+4uNoID9K7TSlE3FdsLtf8snKpMpTjsDB4iSppNJy2jNwHA+QRFevNPqyVbxYnj007uLwXoS0wyH2It2pmN76dOs1AHwPUKqkd3CuS/ymwtwYtZ/5dEOlDoGe8HaEeVdJQ5ENxaQ4o3RZk7oCPQSNoGf5ZwPFuG/0TIMejT3Jq4e5IcxWP2Qc2D3cc4X/UOsL/2HlkNU7tqxP9V/jjQ6Hqk494wWjK9zenYGbFKwf6xluHIKzR0uxXJ/0ZUC1XEe12lNY5/x0LAF+5SJD4HeOA2kUBrRCbLaoQm9F59sr842GwU+1Wgk6XdrsHbNVPhoAFZQvzmnr2GroLPkknWvUOnuoov2F3oATC6EDsDOtwh7sDlC8qqkXPhdGFvFtNwA/Gl1A3aOBpGyfGpi8V7qegJLoXnkSjM4Sx3Haz6c5zrMhGBQ2BKmBhd1RQ0uT7DumD8YZ2hJNCUPUQPv/AV3JtznCX5q7L1q+m8Tl0mbmcXiu0HQfZ4rfZZPBky00STxoXSShsmkVjd4W/d9f4e1Hl08Puemjc+d51NUV1Iqu9caFC4xD76gKDy0sbEMi/wmsrbDFEbQ2vrE8BjaPFhAG9ieh3X4h/14shtu+KaIDWpQtQj2KoSPLWDRzb2VU3R1xrcvDJboboQiqa8ZrPcFPEdkX56W6JGJa11Rxz/CmXtpf8ua6NGB1ybcROW8cofqtTC4xeqXO+ZILAasWig4FvCuwnDqTGqck/zTDom1dFhB0vYbgC7TDo468wSN06AuNxRTaxvJFs++O58hUeSEYExVxZGLHGC2QKl+MGKysTuVzBa7LawmRZW+GLXHU7G1eVN1E1/+4NL6uFCqv2U3tPTqDHZTbo/+PmBpqw2RwXVeJ99c3rn8BGdVbnfpCa6pes1TVJdIeEG59rt52r+8XLXw0vfpzdA/twwdK2wAm+LixoMEVxvscUITYNP8hguK8d/llX0JPF72pAVx30g6QdF1uUP32KCBjIckhhMC+9HvNvP3RfYFoZLwuHW1FddMJZzbv8bJCNval5HFl+2NX07Wvp88Hchg+8ULbrQvgO/vV0/TovjScakhQFC8g7LOgjC+0ARMrASLba+WY78tVXDy4BOqu84LSnXzZE93Vey1yY6qcdiv1OXVe01C59KfqT1uiG+VRVWt0bg7VoqxLPlLKqBn4C64Oru+zb0JJ8WAwFnw5nVq2j5iAUK42ZD7Wez0dF6ZHZam+TCOtOrRaN0KEzSg3fcMfoqSyjLVmOjqDLXRehM9elvXeuS7934Rqlf4jzMTdZq1hH/wuQ0BF/',
    'scripts/features_figures.py': 'eNrNWm1z00gS/u5fMaVk7yROFrbzAnGVt4olYdm6LFBk764oY8RYGtsiekMaBXuB/Xa/7P7YPT0jWbLshBDYq1ORxJ7p6e7p6ZdnWhiG8SSYF5nIWTJjciHYTHCJ7yyXhb+yWRCr0VyuQlGRpDwV2V9zNitXmrmXBanM76sJl7vlhJOuLKfTeX7Jp6L7i0yYl4RJkeU2S2LBIp5dioxhBctFFggMP3AGJPBD4Atmyg9JFwuKSA3IhWWzhyyVTIqltFmcsHkW+HYnSQVUTFLGY59lwXwhWZ4GMbELxVxgsIh9yNCKxyLMHcYe8zAUPpuu2Nu35YZzNxNpkkko/fat0zEMozPLkoi57qygeddlQUQEEBQnkssgifOOpkm5XITBtCJ4ga+dTvklLqJ0xXjO4rQagho+BvAv9auxiMs0TCS4dDr1Z6fIhWk8ms8Na5sQmtInxSeUjO3BKO/5kJ0d9gY7qGXgkb03yTo/nf/jzGbPXz569jP+/vzy7OyZzV6endrsxT9evjjH2MXfX9ns1dn5+fN/KYJXbMRMY6/XezD4aWDYzNg7Oz550uupj73eydmDA/Xx9OjorBx9/PjByaMHht1hd3uMvaPjnw7PThSzJ72zw0Mt+Fg9sM2e8ijtXsoRGs4VJaHwilB0Ll5d/Hb26wXU/0hWCOGVIev+yEy9zi4X2UwGMhSWUtZ4esj6Ts8YMlNbykhI8tN993CfJth//m1Y9pp0oEkrcxpvauLBJvF58JQdaOrS6MYVEZfjG3wHz5l4T5TVkRinJd/BvppqUuYyI1J1hEbeoKMJEH7uPH95evYSVgiDXJqlUaxOp+OLGSJRFqlpkVmewaRDxRfe5WQepnkRyty0GoMveMaj3ClSn0thfjRmSSydPPhdQIeHkM2XSAPKntWgc1QNqwNo0OpwdYhFOfrgBo8xdJJx/DQAZf+oV7HV0e8gJWD8CQ9z0ZpRSaIxl/ozJVSuUhJ6OLhRKm1wxqMgXIHWOBXv+D8LdsFjZWqE24Lyk96EICmGD5KrIieKz2sz8ythYgPIhIUcqoSBlMYjMaRjapkfJE506QeZmfJMgO/ot6yA4mKJA3STS/VVHwpYOsQbf00sY/fZzPhIfD872CU0nE6TpRvE3kLkI0MqQ3x5ZTzfvdJmMP5o0OvVHuGFSa52RjvdY91ul+n6wvpDxGeWiVBlzqqUzDPuB9gTE7kMYD1UEz7nQZxLNS1ikc1XLEw+oELEc+L3xUdbWHuH2xAJ6wVZPkTOdU655E/guKJh/5bRfQSIWjCOUyfIZ0EcwMHViENMXT7NLfYXtj1bKetmJNaaKHb5KpciysF0jLqZoORRodORGMz0N/iL6TuaslyGKdSaavVw7ZcZwjSLq4OzGbk3KYwTyIspZfzc7FMBjM1yLaonKCmuRibKrI18dICxfAGXWpUOlb8vhACBCozSLaAqX9qlBqTl70Fqkrhxb1IN51atGS3wEezz2GabeZUWm6bxyy9dKgpUSFSCghJq8BEG1ynWanBU5sPm/PHaOmw0KkXTEWBYS6Rh/ak0X/XwpUM2Meujg+qtg4LrYmdFipIxQgKGWiEc3ai0H1XFIcpHB3pr2ajaIA/TBR/1nAdH1oZcgiXQdAy5C0Ic8RzwKOPxJWk62K0j1jS0pG931vMYv2feyIjh1dcU30h4611E4sMIJa3eAlTiy6sQSdPsOb1q08beiXogMvyATT/UGgyN1sKFWtj/4sJud3MlwsBdhkFkdnvOMYoF/d4iUCWlKl5j7QuT8WCygxNVGdNo5p79bJ+ZoeA+2ak+GavUQ3s3BE8Uh1XJoZWL7NdxvRTpUvMrWSyAQUIVkmOKyXNYYnBqjhEx9LN5YKpI47QOKzvp0FBSRypcGBLqEnDVS8RsFniUL3OrKvqN58uikpYoHXC1qEfKLln8PYVteqHyOGNvoB5jLVtbj13lLCtiGLpbpEYZxFSZNDQwS7OOyr9YnYCZOhHmQVHYH2UUW6MQnVGCT+JRWeYrWDE6JgSiqplMXI5DhB3g38rTelZdDVWNc0O+QokwM+FJUCka+C1ya78k3SjldgVL+m5V2ZolyL3KXe1GxlZ9HFB9jGe4+sSegDddAZw2S2GSokAGufbg7jc+rRpZiTXl4Lb18bYFTQ6+X0WjqtUuY4cqBaoytmwAIX3TI+3MOpyp3ghcBtRR5AuwdCPhBzyG19DOmvNMzasLBDzydWxqyh2RsBbQVwKkSHOdqWuuj04fvfiNqSlkCrHknmRq7Dpu/VJdfei/o2qIKx4W+tK55n7NdO5XFO2YrRewxoLX8W11Ujv0XHCpddBfa5nM2H/sfnz9mqBwFn3E5OfP++1da1+QAxwQHITnVLPNhulGGAWiysodRYLHYIFBxcWtpzZRCiIfvuCXWYVcUPvBJjypscyWF9Kz8zYIRVvFZgufyMF47ew1QEENwT3cJWuLnMpQFWtu6kljEy6ILKO7NfCCP8F2AS3z9xmhFpnxd3DLhLokluXMgjCMOdVkS8Waz7BHoZytDSnAMsmmPFPQpykadhpjpzjQFWhG+GmDmh24h5JkSCChf/0VSVDZVw0bXeI9nqpwHVS5vnG7LhMCJJLBrgvOYXtTqirnHgcCQPKft6wYJjZbBDAkKOeqfgNLWHTj55KqNPIpNQt84aHk0PF6lzlNpVRGdBtM4H4Sh6sNtlFAV4LqTMKE3YMUa6dqJDAKYpM0oWX3Wb/H7t1D7TiyyK5LcxHomXuNmS1eK44LnuIY8XdJ5qLYcfiAudXTcc6T+Xk5OeW5GPV7N3ILYnBDJIARAmkHv2dFGD5Zzzd4kZ9tHccdoV7bYqoE951trFcCMPW7Ge8UwTozTYbXw75WSd3vd1+/9kUoOTDgD9dAPgU/+z2l0THwgvIeVWN8tqQKBDQ2ZKr1d3LCfoCzqBIdihlqdcKOH9ZjqtmwU4ZyPHN8cmKzE8g5wdRD/Bw/nGxgyDJscuXQaw7k2BoUuSWdq8lM6xrQVPG5ATS1ros34Kd29H8LnDq8BZwaNEAKgaiyjqkasA2lDoasruMqA252FbyEcsDs+8CqDSilhZmUr9tAKhOzfMj8gFJvo+kTCblIfNX2wQlrDO5jFfIznwnjrqgLCnxX3HVTJ+HAObZazYN1Y8HIkg9ljL0LKJvoHIo7sJ9ETtlSdLN4bvbq0NZlfF2lRVxEVO+Fud1ugFXBkWy7VZc9kclgFghK22SPcdMqrQ6CmtFHQTP6U6OJoO9fX+TTYOPgQPHDM5l/COTCNJ4QC4Oo/miTIUpLoiIOKC8bTdEwCIxoM3Ol7LLCZq1Ns4xNo0ZEhnK1cXNkonosCqhVsxq1CckHmJ1Y7ZxeJRslmGDCxjypNN/d3lkbvdnMscsLbP2ewXiz1eIpnXTuiCiVq+FOfIEkIIO4EFuTpPDcwQ3TnYpcOshE6rULyv7fSr9zStNSV6GvkhP9Jreet2olPSvCYfMxWZyQmLJ5g+sWfXKpFMgE9xbCrymnSRJa7BMzldFnQcxDnSOMCftxhGp3LLoHurpIRCItZk8PhyxWrzMAehXHLXFVr2g5Ti4Jwek/d+hXWTew/qPk/ccXmFOqb1/wmy2lnnNifUfIoHtqKhB3xuGkdgNCWjvxxrq9pNEYWCKLka9gaGvB8tYw7Fe+fFbhsACFdQ5L6c5qPEXxGR22PA3ujgAj1XvDXQdxu0ZXYwHp4abqlYyGS9NEyiRqNnSvR3I70BMs2SVLMlT1wBP5zQ2yxi1360pZr+tvrbvFfXG3WAUaT6iV3d8tYOuO0CYoMViPypjGn6rFcDTZIm+D8Zvgs4bOFzKrMbTxA1S4iendMPlX9RlvbP7V1VK3AXkYbvWj7tgHfNOSWxWBUnIR17J1nZ0WPvDtNzY4r+853vAWujLGQkQBfIdxzysy7q0Udiqz+/9tX/LoFkD6oISobRDtEibeRtIX/SH7NP904zs5AGlON+quSjxCqlfuV4J9G5Jet0+1OBN2vvW7O6Kleow/VW+J9B911Yh/1nejp9z6KkCtVv7vXs31/oxXc+V/j9jqZIF8TeOXdhs3d9wCy9qu09ydsx9ZX3SP61EyczU46bQxha9X2cxXhFvB6+jgHZDPN1+FVHClUZtp866tz0fpu1Nd3JOzpEinK3NsAItLygXGpQHr+FmSxrw057ANMBCHVPVG6iWhNy536wT+UkGKib2eoH3U4+3qTm/vGiYZXou2iLC0TbXmK1LbjWgLaigw1NBD/Q8ZNdg8xeu1q0ntmteWfkujxoMVpFu/5tGqPaxVW+qLIEpznnJPmF3sC2nsaPP1o4aiSGrigN3Dmnv32ADYYHCXV4pNMGDf3EZswkSYhjoUjSX16IEYfPubyf1Pczf4tM9M5KSvegPJzKhe8lVIoAy0o51vHOckhjAsNUU503HzZ9Xi9fs/ns0p5GAMmOIbBS9bgON6cWtT3kUivKx2u21/1JL2528G9wf73wQaDv880NC/BWa4aLzMbL7A/C9seHmc',
    'src/fastpath.py': 'eNrdWluP28YVftevmCoPITdcZddJU0COgqRFnBpJHMNxgQUEVZoVh9JgeVsO6ZUS5L/3O2eGd+7GDtqi6MKWyLmcOffbaD6fv5CmVEUg1H2l38lYpaUwpSy1KfXeiCgrxHIfS2OWu1jJItXpYfEDPajwx892wsvy8lKnS7HbRQCUy/K40CkAxLHn73b+YjZ7e1QiL7JIx0pkkchSJf7+7CdhVKz2pc5S4b18eflNIK4/vxKySEwgngWfffGF2GdpqU6l8YU5Zg9GlEdZChknmaGveAZYusSwTpTQRrw+l0cAy96p4qhkuMR6JRSoSGSpAJRe81imRIEw+pBII2Qa8vg+eycLLdO9Iqi9jSLOspyhCoWPs/CAY1AjF4BBcegzoD2QElKYhL5fVcnrsyiyqtSgF1ycKbk/BrzQwilUpEtxW+k4NHxkVMhDQuwHs25jlRjiFsFqGSPBsHfgYiHSrGTWgvAkC6uYaEjyihBWJyyLzwzTSDDnvpJpqUuNOYe2uD03jyQt4tdsdgHsL2J5BtIXwvFhXxUF4RQqsCw1wiOESwhxD26CJ3Y8EHqhFu0o0+aLI3hjgplgLbJUE7sst2oG0imvZRWDEvBjL4uCEKUlEtBTQ5SpKNJ7TXhAJoU+AeRud7OD+rmdJ2aQ/xw0NJIjnTyIT1biavFncfPPtyKpsIPQc2c/H+oD7Qi1PHi0+mcaEjdQYpzW29VqixhqC0HobBafimRwJjBk7hgr+impp/GZOUaQncSJuuc8kOGjMOJOqZzedQHsjhmkTFZ05XeZBYV5OOr9kazjgYyH9xf6oFMZY2GoxAHAWfsSaNO3Vi+ruKGrWazpxBzS0uk+rkJiGasX6follCzHntusgtB2O0v7lytxJ16CeQ+6PGL0DnygLWmV3IIdOMCJrpzG7CGrwJ2jBPE5vI8Khcf8KJKPwaKHdFbrTDpl2VatqlQ7jzDkZb2tFSU81RtlqhiCkYdCAQHCu4dXmYkozuAa08NlnmnIrCCSL7Moeg4KIf7SgEoyRFmoWePggA08WGmVmreAapwakxnyw2I2n89nUZElYruNqrIq1HYrdJJnBXxdCmOXDGg2c2PgYn4W8GBp3ozV7tnCsYZZw3jhtOy1VbLZbPvTm5ffvXz1zQ9iJX79bTabsY8XP7D1L6FTQgCl1+Dc0GKXzuiEDskjFYl9uFnnNMojG59dV40DxFQuiEAGG6oIROpUl9utBx5FQe1cfHH5lXiF+GDPpz+aX6RbFtkKFKaeW7u+2vj9VYQnLVpvmnEyoohE7V0hvPgtWPrbl6cttDcQjLd9dCM4jh4YWND539tPwDkUeIDkoACCT+cpyEcVcAgNutFmcDz96YhJwn5frOCnxisYUfBdp5UaTToKFjLPVRp6ab6QBu5Tni1GYXnO1Qqj0NQvPvf90f6G8AkIjpzfhfEYBs35bDETGx2PO1ujCjlDzY/A8vbJ8w+wpdxYBR6w1bFmOcUzbABAcHUPAaUkJLf6EQ6N1zeMm+DHePUjsMlyRmsdV8arsyKELfL6WJ0MbNqziYgTFPHMn6K21czAHYnBNYPbuLnmrR3HSvc8goncriiNRSWKZZlm6S+qyEiCoY4ilp4vPhHXo51kMXEWiKMmG/lF596A+jWsOnAHbKACw2k7A0vEGQujf1Gbje9PG00OGgLxDmhaEuNsedSOsPr55J6mAXB4CSgsP1hiEU3uKwg/QACHh07hwxBZjFq9LSrlTwKps7Ys7oGQHwDixu4kFhvPap5h2pkW++g/snVNyNvzNwDzbnKVtaI1DIxEB5ZsfFrs1QywNNz0j2g9bm3AFoyPSMIO3qaRE/69G2Hu1JlO0mHXqyPHaN+v6X3g9i0me6TTCrsZk+021HvEk4VRJY4H5qU331JB4hAx8wBewu48Is1c2f2Lgyo9YGEn4DdoDkkFAi4HIo57GMOxNNwi0Uxc9yauN602WgnzomdWxSwqONvyoKapOZx9H6Hli69W4tlSiI8QRN0uBFSqmkp5R9mxiNSDOCKTKJAY/fjXZZMPDhJ2xpMISbKiDSCW9jyDASKgexr1nzu4y9w1GMOa0JIddCgNHGZ2h6PWDtVK0KTFTg9Qqai4rw5Q7TTkkLHs82hag3jNoWsSdpnLEez8o2GfQ3acH2F/I/Wm9Q4W6zUi9gKMSYw38DBJheOZkkWiZGo8B7EGeC0uRdS3FoqLYbiQpXeoD6Sa5EJ4N4u34muA9HtcPNQMpKx2azPaSRbWFeEfZqVLiVeco3kOmEUle9hC56yb58jcZXQgLq9H0bmzb91JBSzU0eqNhS0LmR4Ux313vIVjFFx/2HN+nTVBnRk2TQK3i/Ns5G1WQPzqXfmOb+792rf2WB4LZahGrd0IgEX6sChkqCuzTTR4TxueVqqjsgBo5ZqFv/n3KlxBzY9VzVja3o9YbPoruwx+42qYC5EzozULmZ69iXA5mV/eSIC8WS8D3roZwiSi10zZRnzZMnIM/CN2SrY0RD1oq6o7wO4Vc+OqkOsz3TQsRnAJhAck/wSC/YWpEk+etFldjSPhvaBK9IJXX1yIZ4+vVrFRYwrY/BqTbwvG97D7+niosNIpHTvPZZDfB/fy8iuJkHSD/QyfHocRlvR/TVJdswgC4fjtOhoXBDYBHQng3pNHueo5Ecya+4KsngA1gbklYGt7KU86lsB2k5aQIOcwLyR49H/obQou/R/3Nt0Nf8DTZIXcx2pLjoXKCeYpdfGGXkdTGTi3q+f/jVg2cFzOpIfWnlA35+o9Pcf/nLuqqHnU9RYEmO3/2q8d1OOdooC6LL02kGvZ6nKAAUy5uiU05JqOrKsrY9+GZHZV4nGnwwpil064Hpz3IR7MpsrkurdkoM7iW1jOhwU9legf8D3Tx0mLhfo1sf5TkfTJC09OwuuxOKypkcnq0xa5/4kS/pPv/Nozcmzfd/2YXV+7L2p1bp1TYRfjNITGzagcoPTe1QG8YN4m+rweAabfbyLbJ/8kk9tQUgWZLwcuIXfeIIcjIO1Js3u5FN/+5bNrIbzbONvfGWgT61zdw+X2rMW55WaN8KBdpQf9JFS8IMIMukpMpcljXcKfo043AzshAxu0/DzbJBgCdolUrG8LWZwXhFXAXOjL/dSqDwngBPKnjPrEFrbo2zRTWpeJHpX0wHhLtaaLa/WQrdTdoJv1B12+jjBxHn93lcUOzGbbf7x6+dOrn5eCFKFucbr4xwnFli0eKtRlIsU12kDFcNCJcJteQ7R3c0GdGIi628uGe3GZDd1rCYatU75LQj6/VyRKbRuibRGMmreHSqdSdbR8YK3ahTZZlF5bxZtsiHUD6n4US1lRMbzFPyKme5TzF480zrpw82m4AJo/DneqDTbqgLVq/kS3a/bhvSyrMm2/8T/dz7LnjToznVZUp7VlSa91ZU+XAV5vrCnnu2wN3CG9zNEONXcC2zd0X/W3Jvo0xrDbDXPi3Y6ZQtrIl1xBfSsZtneDHqUkzQWqjwl3E8PvH5s6FtelwMJeG9jLKfgzfLrbKXMsdHonD91rHFJ9KaIqtVfL7o6YSoxc6m6BQbdskE1GkKwuVLeX7L9r8IwH8LFpMgG+1eUl6LvEN69wM/3LK7o4sIw0T156uPDr+P/I1QcvqouP/lSjjU5c/dYccmjr/NZllcfKejR8bHp+zXnGFkPCvsGQM4Ol4K02lVl2Nturb56eLAg6DT4rcQbXunNCsY4q9Nz3b43+10ucMQwSVLhBhjMK5MMTHqveGKy7xZ/ocAJCbTf0POtnOFRiwc72RxsFPAfM8sofOl0GNkp76rjE9tJXjU5vhwqmNX24UARCKDIMLLNO6UZBbpQuPFnDwEWTY/bs1c2wounPOj9a5bkNhMyUBIpOVanNI9kH2oxyw53NbtVDTRZifqfTwveYEjkLYQZoD0dVIOWpjwAwW+yyq+FOAe/d9PLAiUTRqbn9HQNAcxHK6ymla18cSYTkoZLIS+jmW9SVCsNNZF4rVjdzaqICBwSnFE6KrWpOZGE1sK6M1p2LiY/sjwJcz8TlGYk8W16JVKmpHxJwr8WmpLNBgdVwc91GEwp2m6mqa1RxNVet75tj1kwLupk1rG4VLUmVPfc6dY8xKIDf83IX0PZHtacuUV0BB+5p0KKrZfo7Ce7JZZv9Uxb1MNQ5P3v+YDbLS51oQ/0Gp9LrYUGVL7amLKo9XfmbXthum/VpqEPkC3SXtThNlctdg2KVpd6CN66ciSOLxll0q7ygPcYXXz69kujtozhRpZ/4isWBHFekA+NC6Xf93vVEe6UOE+Gbqg8sKcZ+fssEb1Gxej3oHRWO5nR7+qv+zeqR+DX6bd7LmtxG58+b38D1AzsSgjfKVgGcwcTZg0hUecxC/rXXk7+006FK8gy5Y+k3xQMUoPklxzDLb7xRKqmoi1I2FtTDzbXMPOjc0cAJYq5z40CznVc/mLw+pE2jnuI8mGg02gM4ztG8/Y1YR3UaQtaEMEkWnJdlWXhjdgRMVLc2fHolkV+3QOHupoUz5FWD0Lh59n7n9ahqjPtfzGqpAg==',
}
for name, blob in FILES.items():
    path = REPO / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(zlib.decompress(base64.b64decode(blob)))
    print('wrote', path.relative_to(ROOT))

In [ ]:
# ---- helpers ------------------------------------------------------------------------------
import time, shlex, json
from concurrent.futures import ThreadPoolExecutor, as_completed

os.chdir(REPO)
FEAT = REPO / "runs" / "features"
LOGS = FEAT / "logs"
LOGS.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
PY = sys.executable

def run_parallel(jobs, workers=None):
    """jobs: list of (name, argv); one core each; output in runs/features/logs/<name>.log"""
    workers = workers or WORKERS
    if not jobs:
        print("nothing to do (already on disk)"); return {}
    if DRY_RUN:
        for name, argv in jobs:
            print(name, ":", " ".join(shlex.quote(a) for a in argv))
        return {}
    print(f"{len(jobs)} jobs on {workers} workers; logs in {LOGS}", flush=True)
    t0, codes = time.time(), {}
    def one(name, argv):
        with open(LOGS / f"{name}.log", "w") as log:
            return subprocess.call(argv, stdout=log, stderr=subprocess.STDOUT, env=ENV, cwd=REPO)
    with ThreadPoolExecutor(workers) as pool:
        futures = {pool.submit(one, n, a): n for n, a in jobs}
        for k, f in enumerate(as_completed(futures), 1):
            name = futures[f]; codes[name] = f.result()
            status = "ok" if codes[name] == 0 else f"FAILED (exit {codes[name]}): see logs/{name}.log"
            print(f"[{k}/{len(jobs)}] {name}: {status}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
    return codes

## 1. Build the H2O problems (once; a few minutes each, cached in `.cache`)

In [ ]:
run_parallel([(f"build_{c}", [PY, "scripts/build_problems.py", "--cases", c]) for c in CASES])

## 2. Table 1 — correlation of the gradient estimates vs energy lowering

One job for the HF and CISD states of each geometry, and one job per exact ADAPT state.

In [ ]:
EXACT_STEPS = {"H2O_eq_HF": 17, "H2O_stretch_HF": 18}   # exact ADAPT to chemical accuracy (STO-3G)
jobs = []
for case in CASES:
    parts = [("fixed", ["--states", "HF", "CISD"])]
    parts += [(f"k{k}", ["--states", "ADAPT", "--ks", str(k)]) for k in range(1, EXACT_STEPS[case])]
    for tag, extra in parts:
        name = f"gradient_correlation_pairs_h2o_{case}_{tag}.csv"
        if (FEAT / name).exists():
            continue
        jobs.append((f"table1_{case}_{tag}",
                     [PY, "scripts/features_gradient_correlation.py", "--cases", case,
                      "--levels", *TABLE1_LEVELS, "--name", name, *extra]))
run_parallel(jobs)

## 3. Table 2 — confidence level vs optimization

**3a** (minutes): the 95 % level from the 36 recorded runs per geometry; the exact optimization is
recomputed along their choices and checked against the recorded energies.
**3b** (hours): new measured runs at the other levels. Run the cell again until no job is left.

In [ ]:
run_parallel([(f"replay_{c}", [PY, "scripts/features_replay.py", "--case", c, "--skip-done",
                               "--patterns", "II_A_data__safe", "II_0_safe", "M2_safe", "M1_seq__safe"])
              for c in CASES], workers=min(WORKERS, len(CASES)))

In [ ]:
def table2_jobs():
    jobs = []
    for case in CASES:
        for d in DELTAS:
            for i in range(RUNS_PER_LEVEL):
                out = FEAT / case / f"ci_II-A_data_safe_d{d:g}_i{i}.jsonl"
                if out.exists() and out.read_text().strip():
                    continue
                jobs.append((f"table2_{case}_d{d:g}_i{i}",
                             [PY, "scripts/features_ci_sweep.py", "--case", case, "--methods", "II-A data, safe",
                              "--deltas", str(d), "--trajectories", str(RUNS_PER_LEVEL), "--only", str(i),
                              "--fast", "--checkpoint", "--max-hours", str(MAX_HOURS)]))
    return jobs

def table2_status():
    done = [(c, d, i) for c in CASES for d in DELTAS for i in range(RUNS_PER_LEVEL)
            if (FEAT / c / f"ci_II-A_data_safe_d{d:g}_i{i}.jsonl").exists()
            and (FEAT / c / f"ci_II-A_data_safe_d{d:g}_i{i}.jsonl").read_text().strip()]
    total = len(CASES) * len(DELTAS) * RUNS_PER_LEVEL
    print(f"Table 2 runs finished: {len(done)} / {total}")
    return total - len(done)

jobs = table2_jobs()
if MODE == "local":
    run_parallel(jobs, workers=WORKERS_TABLE2)
    if table2_status():
        print("  -> run this cell again to continue the unfinished runs")
elif jobs:
    tasks = FEAT / "slurm_tasks.txt"
    tasks.write_text("".join(" ".join(shlex.quote(a) for a in argv) + f" > {shlex.quote(str(LOGS / name))}.log 2>&1\n"
                             for name, argv in jobs))
    script = FEAT / "slurm_table2.sh"
    script.write_text("\n".join([
        "#!/bin/bash",
        "#SBATCH --job-name=h2o-table2",
        f"#SBATCH --array=1-{len(jobs)}",
        "#SBATCH --cpus-per-task=1",
        f"#SBATCH --mem={SLURM_MEM}",
        f"#SBATCH --time={SLURM_TIME}",
        f"#SBATCH --output={LOGS}/slurm-%A_%a.out",
        *[f"#SBATCH {x}" for x in SLURM_EXTRA],
        f"cd {shlex.quote(str(REPO))}",
        "export OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1",
        f'sed -n "${{SLURM_ARRAY_TASK_ID}}p" {shlex.quote(str(tasks))} | bash', ""]))
    if DRY_RUN:
        print(script.read_text())
    else:
        print(subprocess.check_output(["sbatch", str(script)], text=True))
        print("submitted; check progress with the next cell")
else:
    table2_status()

**Status of the Table 2 runs.** Run this cell at any time. If a run hit the time limit before it
finished, run the cell above again: it submits only the unfinished runs, and they continue from their checkpoints.

In [ ]:
left = table2_status()
if MODE == "slurm" and not DRY_RUN:
    print(subprocess.run(["squeue", "--me", "--name=h2o-table2"], text=True, capture_output=True).stdout)

## 4. Table 3 — selection errors vs optimization cost (fixed budgets of the recorded runs)

In [ ]:
jobs = []
for case, budgets in FIXED_BUDGETS.items():
    if case not in CASES:
        continue
    for b in budgets:
        if (FEAT / case / f"ci_Fixed_{b}_pilot_d0.05.jsonl").exists():
            continue
        jobs.append((f"table3_{case}_{b}",
                     [PY, "scripts/features_replay.py", "--case", case, "--skip-done", "--patterns", "fixed_fine_",
                      "--exclude", "reuse", "uniform", "designed", "--methods", f"Fixed {b}, pilot",
                      "--max-runs", str(FIXED_MAX_RUNS)]))
run_parallel(jobs)

## 5. The three tables and their figures

Files in `runs/features/report/`: `table1_*`, `table2_*`, `table3_*` (`.tex` and `.csv`) and `figure1-3_*` (`.png`, `.pdf`).

In [ ]:
subprocess.call([PY, "scripts/features_report.py"], cwd=REPO, env=ENV)
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
R = FEAT / "report"
for title, f in [("Table 1: gradient-estimate correlation vs energy lowering", "table1_gradient_correlation_vs_energy.csv"),
                 ("Table 2: confidence level vs optimization", "table2_confidence_vs_optimization.csv"),
                 ("Table 3: selection errors vs optimization cost", "table3_errors_vs_optimization_cost.csv")]:
    print(title)
    display(pd.read_csv(R / f).round(3)) if (R / f).exists() else print("  (no data yet)")
for f in ["figure1_gradient_correlation_vs_energy", "figure2_confidence_vs_optimization",
          "figure3_errors_vs_optimization_cost"]:
    if (R / f"{f}.png").exists():
        display(Image(filename=str(R / f"{f}.png")))
print("files:", *sorted(p.name for p in R.glob("table*")), sep="\n  ")